In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

# =============================================================================
# JFAMP — FINAL OPTIMIZER PREFLIGHT
#
# Purpose:
#   1. Discover the saved experiment source mounted in Kaggle.
#   2. Verify all necessary experiment stages and artifacts.
#   3. Validate CSV schemas and important row counts.
#   4. Verify C.4.3 restoration completeness.
#   5. Establish paths for the final optimizer.
#
# This cell performs NO model inference and does not modify source artifacts.
#
# Expected:
#   All required stages present.
#   371 layer-format combinations.
#   200 C.4.2 fault trials.
#   8 catastrophic trials.
#   57 C.4.3 restoration evaluations.
# =============================================================================

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

INPUT_ROOT = Path("/kaggle/input")

EXPERIMENT_FOLDER = "revised_jfamp_fault_aware"

EXPECTED_CANDIDATE = "CAND_01269"

EXPECTED_LAYER_COUNT = 53
EXPECTED_FORMAT_COUNT = 7
EXPECTED_LAYER_FORMAT_ROWS = 371

EXPECTED_C42_TRIALS = 200
EXPECTED_CATASTROPHIC_TRIALS = 8
EXPECTED_C43_RESTORATIONS = 57

print("=" * 80)
print("JFAMP FINAL OPTIMIZER — INPUT PREFLIGHT")
print("=" * 80)

print("Kaggle input root:", INPUT_ROOT)
print("Input root exists:", INPUT_ROOT.exists())

if not INPUT_ROOT.exists():
    raise RuntimeError("Kaggle input directory not found.")

# =============================================================================
# 2. DISCOVER EXPERIMENT ROOT
# =============================================================================

print("\nSearching mounted Kaggle sources...")

candidate_roots = []

for path in INPUT_ROOT.rglob(EXPERIMENT_FOLDER):

    if path.is_dir():
        candidate_roots.append(path)

if not candidate_roots:
    raise FileNotFoundError(
        f"Could not find '{EXPERIMENT_FOLDER}' under /kaggle/input. "
        "Attach the saved experiment output as a Kaggle source."
    )

print("Experiment directory candidates:", len(candidate_roots))

# Required high-value artifacts used to rank duplicate copies.
ranking_files = [
    "C1_layer_format_optimizer_dataset.csv",
    "C2_candidate_layer_allocations.csv",
    "C3_clean_candidate_results.csv",
    "C4_2_fault_trials.csv",
    "C4_3_individual_bit_restorations.csv",
]

ranked = []

for root in candidate_roots:

    score = sum(
        (root / filename).is_file()
        for filename in ranking_files
    )

    ranked.append((score, root))

ranked.sort(
    key=lambda item: item[0],
    reverse=True
)

best_score = ranked[0][0]

if best_score == 0:
    raise RuntimeError(
        "Found experiment directories, but none contains "
        "the required optimizer artifacts."
    )

best_candidates = [
    root for score, root in ranked
    if score == best_score
]

if len(best_candidates) > 1:

    print("\nMultiple equally complete experiment sources found:")

    for root in best_candidates:
        print(root)

    raise RuntimeError(
        "Ambiguous experiment sources. Detach duplicate sources "
        "or keep only one complete saved experiment source."
    )

EXPERIMENT_ROOT = best_candidates[0]
ROOT = EXPERIMENT_ROOT

print("\nSelected experiment root:")
print(EXPERIMENT_ROOT)

# =============================================================================
# 3. DEFINE STAGE DIRECTORIES
# =============================================================================

STAGE_DIRS = {

    "B": "stage_B_layerwise_fault_sensitivity",

    "B2": "stage_B2_sensitivity_matrices",

    "B3": "stage_B3_full_model_validation",

    "B3.4": "stage_B3_4_final_statistical_validation",

    "C1": "stage_C1_optimizer_dataset",

    "C2": "stage_C2_multiobjective_optimizer",

    "C3": "stage_C3_clean_full_model_validation",

    "C4": "stage_C4_fault_validation",

    "C4.2": "stage_C4_2_extended_fault_validation",

    "C4.3": "stage_C4_3_final_fault_attribution",
}

STAGES = {
    key: EXPERIMENT_ROOT / value
    for key, value in STAGE_DIRS.items()
}

# =============================================================================
# 4. REQUIRED FILES
# =============================================================================

REQUIRED_FILES = {

    "C1": [
        "C1_layer_format_optimizer_dataset.csv",
        "C1_B3_full_model_benchmarks.csv",
        "C1_B3_fault_benchmark_summary.csv",
        "C1_B34_coverage_audit.csv",
        "C1_dataset_manifest.json",
    ],

    "C2": [
        "C2_candidate_objectives.csv",
        "C2_candidate_layer_allocations.csv",
        "C2_pareto_frontier.csv",
        "C2_optimizer_manifest.json",
    ],

    "C3": [
        "C3_clean_candidate_results.csv",
        "C3_experiment_manifest.json",
    ],

    "C4": [
        "C4_fault_trials.csv",
        "C4_fault_summary.csv",
        "C4_experiment_manifest.json",
    ],

    "C4.2": [
        "C4_2_fault_trials.csv",
        "C4_2_fault_summary.csv",
        "C4_2_catastrophic_fault_attribution.csv",
        "C4_2_experiment_manifest.json",
    ],

    "C4.3": [
        "C4_3_catastrophic_trial_reproduction.csv",
        "C4_3_individual_bit_restorations.csv",
        "C4_3_trial_recovery_summary.csv",
        "C4_3_vulnerable_layer_summary.csv",
        "C4_3_vulnerable_format_summary.csv",
        "C4_3_final_evidence_summary.json",
        "C4_3_experiment_manifest.json",
    ],
}

# =============================================================================
# 5. FILE AND DIRECTORY AUDIT
# =============================================================================

audit_rows = []

for stage, dirname in STAGE_DIRS.items():

    directory = STAGES[stage]

    exists = directory.is_dir()

    audit_rows.append({
        "stage": stage,
        "artifact": dirname,
        "type": "directory",
        "exists": exists,
        "size_bytes": None,
    })

    if not exists:
        continue

    if stage in REQUIRED_FILES:

        for filename in REQUIRED_FILES[stage]:

            filepath = directory / filename

            audit_rows.append({
                "stage": stage,
                "artifact": filename,
                "type": "file",
                "exists": filepath.is_file(),
                "size_bytes": (
                    filepath.stat().st_size
                    if filepath.is_file()
                    else None
                ),
            })

    else:

        csv_files = list(directory.glob("*.csv"))

        audit_rows.append({
            "stage": stage,
            "artifact": "at least one CSV result",
            "type": "content",
            "exists": len(csv_files) > 0,
            "size_bytes": None,
        })

audit_df = pd.DataFrame(audit_rows)

print("\n--- STAGE ARTIFACT AUDIT ---")
print(audit_df.to_string(index=False))

missing_artifacts = audit_df[
    audit_df["exists"] == False
]

if len(missing_artifacts):

    print("\nMISSING ARTIFACTS:")
    print(missing_artifacts.to_string(index=False))

# =============================================================================
# 6. MODEL CHECKPOINT AND CANONICAL REFERENCE DISCOVERY
# =============================================================================

print("\n--- MODEL AND VALIDATION REFERENCES ---")

checkpoint_candidates = list(
    INPUT_ROOT.rglob(
        "mobilenetv2_cifar100_best.pth"
    )
)

reference_candidates = list(
    INPUT_ROOT.rglob(
        "fp32_reference_predictions.npz"
    )
)

print("Model checkpoint candidates:", len(checkpoint_candidates))
print("Reference prediction candidates:", len(reference_candidates))

if not checkpoint_candidates:
    print("WARNING: Original model checkpoint not found.")

if not reference_candidates:
    print("WARNING: Canonical FP32 reference predictions not found.")

MODEL_CHECKPOINT = (
    checkpoint_candidates[0]
    if len(checkpoint_candidates) == 1
    else None
)

REFERENCE_FILE = (
    reference_candidates[0]
    if len(reference_candidates) == 1
    else None
)

if len(checkpoint_candidates) > 1:

    print("Multiple checkpoint copies detected.")
    print("Their hashes will be compared.")

if len(reference_candidates) > 1:

    print("Multiple reference copies detected.")
    print("Their hashes will be compared.")

def file_sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


if len(checkpoint_candidates):

    checkpoint_hashes = {
        file_sha256(p)
        for p in checkpoint_candidates
    }

    print("Checkpoint unique SHA-256 values:", len(checkpoint_hashes))

    if len(checkpoint_hashes) == 1:
        MODEL_CHECKPOINT = checkpoint_candidates[0]
        print("Checkpoint copies are identical.")

if len(reference_candidates):

    reference_hashes = {
        file_sha256(p)
        for p in reference_candidates
    }

    print("Reference unique SHA-256 values:", len(reference_hashes))

    if len(reference_hashes) == 1:
        REFERENCE_FILE = reference_candidates[0]
        print("Reference copies are identical.")

# =============================================================================
# 7. VALIDATE C1 UNIFIED OPTIMIZER DATASET
# =============================================================================

print("\n--- C1 OPTIMIZER DATASET VALIDATION ---")

C1_DATASET = (
    STAGES["C1"]
    / "C1_layer_format_optimizer_dataset.csv"
)

c1_df = pd.read_csv(C1_DATASET)

required_c1_columns = [
    "layer",
    "format",
]

missing_c1_columns = [
    c for c in required_c1_columns
    if c not in c1_df.columns
]

if missing_c1_columns:
    raise RuntimeError(
        f"C1 missing columns: {missing_c1_columns}"
    )

layer_count = c1_df["layer"].nunique()
format_count = c1_df["format"].nunique()

print("C1 rows:", len(c1_df))
print("Unique layers:", layer_count)
print("Unique formats:", format_count)
print("Layer-format duplicates:", c1_df.duplicated(
    ["layer", "format"]
).sum())

if len(c1_df) != EXPECTED_LAYER_FORMAT_ROWS:
    print("WARNING: Unexpected C1 row count.")

if layer_count != EXPECTED_LAYER_COUNT:
    print("WARNING: Unexpected layer count.")

if format_count != EXPECTED_FORMAT_COUNT:
    print("WARNING: Unexpected format count.")

# =============================================================================
# 8. VALIDATE C2 CANDIDATE ALLOCATIONS
# =============================================================================

print("\n--- C2 CANDIDATE VALIDATION ---")

C2_OBJECTIVES = (
    STAGES["C2"] / "C2_candidate_objectives.csv"
)

C2_ALLOCATIONS = (
    STAGES["C2"] / "C2_candidate_layer_allocations.csv"
)

C2_PARETO = (
    STAGES["C2"] / "C2_pareto_frontier.csv"
)

objectives_df = pd.read_csv(C2_OBJECTIVES)
allocations_df = pd.read_csv(C2_ALLOCATIONS)
pareto_df = pd.read_csv(C2_PARETO)

required_c2_columns = [
    "candidate_id",
    "layer",
    "format",
]

missing_c2_columns = [
    c for c in required_c2_columns
    if c not in allocations_df.columns
]

if missing_c2_columns:
    raise RuntimeError(
        f"C2 allocation columns missing: {missing_c2_columns}"
    )

candidate_ids = set(
    allocations_df["candidate_id"].astype(str)
)

print("Candidate allocations:", len(allocations_df))
print("Unique candidate IDs:", len(candidate_ids))
print("CAND_01269 present:", EXPECTED_CANDIDATE in candidate_ids)

if EXPECTED_CANDIDATE not in candidate_ids:
    raise RuntimeError(
        "Previously validated candidate CAND_01269 is missing."
    )

candidate_allocation = allocations_df[
    allocations_df["candidate_id"].astype(str)
    == EXPECTED_CANDIDATE
].copy()

print(
    "CAND_01269 layer allocations:",
    len(candidate_allocation)
)

if len(candidate_allocation) != EXPECTED_LAYER_COUNT:
    print("WARNING: CAND_01269 layer count mismatch.")

# =============================================================================
# 9. VALIDATE C3 CLEAN FULL-MODEL RESULTS
# =============================================================================

print("\n--- C3 CLEAN VALIDATION ---")

C3_RESULTS = (
    STAGES["C3"] / "C3_clean_candidate_results.csv"
)

c3_df = pd.read_csv(C3_RESULTS)

print("C3 candidate records:", len(c3_df))
print("C3 columns:", list(c3_df.columns))

if "candidate_id" in c3_df.columns:

    c3_candidate = c3_df[
        c3_df["candidate_id"].astype(str)
        == EXPECTED_CANDIDATE
    ]

    print(
        "CAND_01269 clean-validation records:",
        len(c3_candidate)
    )

else:

    print(
        "WARNING: candidate_id column not found in C3 results."
    )

# =============================================================================
# 10. VALIDATE C4.2 EXTENDED FAULT TRIALS
# =============================================================================

print("\n--- C4.2 EXTENDED FAULT VALIDATION ---")

C42_TRIALS = (
    STAGES["C4.2"] / "C4_2_fault_trials.csv"
)

C42_SUMMARY = (
    STAGES["C4.2"] / "C4_2_fault_summary.csv"
)

C42_ATTRIBUTION = (
    STAGES["C4.2"]
    / "C4_2_catastrophic_fault_attribution.csv"
)

c42_df = pd.read_csv(C42_TRIALS)
c42_summary_df = pd.read_csv(C42_SUMMARY)
c42_attribution_df = pd.read_csv(C42_ATTRIBUTION)

c42_candidate = c42_df[
    c42_df["candidate_id"].astype(str)
    == EXPECTED_CANDIDATE
].copy()

c42_successful = c42_candidate[
    c42_candidate["status"].astype(str) == "OK"
]

c42_catastrophic = c42_successful[
    c42_successful["accuracy_degradation_pp"].astype(float)
    >= 10.0
]

print("C4.2 total records:", len(c42_candidate))
print("C4.2 successful trials:", len(c42_successful))
print("C4.2 catastrophic trials:", len(c42_catastrophic))
print("C4.2 attribution rows:", len(c42_attribution_df))

if len(c42_successful) != EXPECTED_C42_TRIALS:
    print("WARNING: C4.2 expected 200 successful trials.")

if len(c42_catastrophic) != EXPECTED_CATASTROPHIC_TRIALS:
    print("WARNING: C4.2 expected 8 catastrophic trials.")

# =============================================================================
# 11. VALIDATE C4.3 FINAL ATTRIBUTION
# =============================================================================

print("\n--- C4.3 FINAL ATTRIBUTION VALIDATION ---")

C43_REPRODUCTION = (
    STAGES["C4.3"]
    / "C4_3_catastrophic_trial_reproduction.csv"
)

C43_RESTORATIONS = (
    STAGES["C4.3"]
    / "C4_3_individual_bit_restorations.csv"
)

C43_TRIAL_RECOVERY = (
    STAGES["C4.3"]
    / "C4_3_trial_recovery_summary.csv"
)

C43_LAYER_SUMMARY = (
    STAGES["C4.3"]
    / "C4_3_vulnerable_layer_summary.csv"
)

C43_FORMAT_SUMMARY = (
    STAGES["C4.3"]
    / "C4_3_vulnerable_format_summary.csv"
)

C43_EVIDENCE = (
    STAGES["C4.3"]
    / "C4_3_final_evidence_summary.json"
)

C43_MANIFEST = (
    STAGES["C4.3"]
    / "C4_3_experiment_manifest.json"
)

c43_reproduction_df = pd.read_csv(
    C43_REPRODUCTION
)

c43_restoration_df = pd.read_csv(
    C43_RESTORATIONS
)

c43_recovery_df = pd.read_csv(
    C43_TRIAL_RECOVERY
)

c43_layer_df = pd.read_csv(
    C43_LAYER_SUMMARY
)

c43_format_df = pd.read_csv(
    C43_FORMAT_SUMMARY
)

with open(C43_EVIDENCE, "r") as f:
    c43_evidence = json.load(f)

with open(C43_MANIFEST, "r") as f:
    c43_manifest = json.load(f)

print(
    "C4.3 reproduced catastrophic trials:",
    len(c43_reproduction_df)
)

print(
    "C4.3 restoration rows:",
    len(c43_restoration_df)
)

print(
    "C4.3 trial recovery rows:",
    len(c43_recovery_df)
)

print(
    "C4.3 layer summary rows:",
    len(c43_layer_df)
)

print(
    "C4.3 format summary rows:",
    len(c43_format_df)
)

# Use the canonical name global_bit_position.
if "global_bit_position" not in c43_restoration_df.columns:
    raise RuntimeError(
        "C4.3 restoration CSV missing global_bit_position."
    )

c43_successful = c43_restoration_df[
    c43_restoration_df["status"].astype(str) == "OK"
]

c43_failed = c43_restoration_df[
    c43_restoration_df["status"].astype(str) != "OK"
]

print("Successful restorations:", len(c43_successful))
print("Failed restorations:", len(c43_failed))

print(
    "Exact prediction restorations:",
    c43_successful[
        "exact_clean_predictions_restored"
    ].astype(str).str.lower().isin(
        ["true", "1"]
    ).sum()
)

if len(c43_reproduction_df) != EXPECTED_CATASTROPHIC_TRIALS:
    print("WARNING: C4.3 reproduction count mismatch.")

if len(c43_successful) != EXPECTED_C43_RESTORATIONS:
    print(
        "WARNING: C4.3 does not yet contain all 57 "
        "successful restoration evaluations."
    )

if len(c43_failed):
    print("WARNING: Failed C4.3 restorations are present.")

# =============================================================================
# 12. VALIDATE STAGE B, B2, B3 AND B3.4 CONTENT
# =============================================================================

print("\n--- EARLIER EXPERIMENT STAGE CONTENT ---")

for stage in ["B", "B2", "B3", "B3.4"]:

    directory = STAGES[stage]

    csv_files = list(directory.glob("*.csv"))
    json_files = list(directory.glob("*.json"))
    png_files = list(directory.glob("*.png"))

    print(
        f"{stage:5s} | "
        f"CSV={len(csv_files):3d} | "
        f"JSON={len(json_files):3d} | "
        f"PNG={len(png_files):3d}"
    )

    if len(csv_files) == 0:
        print(
            f"WARNING: No CSV results found for stage {stage}."
        )

# =============================================================================
# 13. VALIDATE MANIFESTS AND EXPERIMENT PROVENANCE
# =============================================================================

print("\n--- MANIFEST VALIDATION ---")

manifest_paths = [
    STAGES["C1"] / "C1_dataset_manifest.json",
    STAGES["C2"] / "C2_optimizer_manifest.json",
    STAGES["C3"] / "C3_experiment_manifest.json",
    STAGES["C4"] / "C4_experiment_manifest.json",
    STAGES["C4.2"] / "C4_2_experiment_manifest.json",
    STAGES["C4.3"] / "C4_3_experiment_manifest.json",
]

for path in manifest_paths:

    if path.is_file():

        try:

            with open(path, "r") as f:
                data = json.load(f)

            print(
                "[OK]",
                path.name,
                "| keys:",
                len(data)
            )

        except Exception as exc:

            print(
                "[INVALID]",
                path.name,
                "|",
                exc
            )

# =============================================================================
# 14. FINAL PREFLIGHT DECISION
# =============================================================================

missing_files = audit_df[
    (audit_df["type"] == "file")
    & (~audit_df["exists"])
]

missing_directories = audit_df[
    (audit_df["type"] == "directory")
    & (~audit_df["exists"])
]

critical_checks = {

    "experiment_root_found": EXPERIMENT_ROOT.is_dir(),

    "C1_dataset_371_rows": (
        len(c1_df) == EXPECTED_LAYER_FORMAT_ROWS
    ),

    "C1_53_layers": (
        layer_count == EXPECTED_LAYER_COUNT
    ),

    "C1_7_formats": (
        format_count == EXPECTED_FORMAT_COUNT
    ),

    "CAND_01269_allocation_present": (
        len(candidate_allocation) == EXPECTED_LAYER_COUNT
    ),

    "C3_candidate_result_present": (
        len(c3_candidate) >= 1
        if "candidate_id" in c3_df.columns
        else False
    ),

    "C4_2_200_successful_trials": (
        len(c42_successful) == EXPECTED_C42_TRIALS
    ),

    "C4_2_8_catastrophic_trials": (
        len(c42_catastrophic) == EXPECTED_CATASTROPHIC_TRIALS
    ),

    "C4_3_8_reproductions": (
        len(c43_reproduction_df) == EXPECTED_CATASTROPHIC_TRIALS
    ),

    "C4_3_57_successful_restorations": (
        len(c43_successful) == EXPECTED_C43_RESTORATIONS
    ),

    "C4_3_zero_failed_restorations": (
        len(c43_failed) == 0
    ),

    "model_checkpoint_present": (
        MODEL_CHECKPOINT is not None
    ),

    "canonical_reference_present": (
        REFERENCE_FILE is not None
    ),

    "all_required_named_files_present": (
        len(missing_files) == 0
    ),

    "all_required_stage_directories_present": (
        len(missing_directories) == 0
    )
}

print("\n" + "=" * 80)
print("FINAL PREFLIGHT CHECKLIST")
print("=" * 80)

for name, passed in critical_checks.items():

    print(
        f"[{'PASS' if passed else 'FAIL'}] {name}"
    )

preflight_passed = all(
    critical_checks.values()
)

# =============================================================================
# 15. EXPORT PATHS FOR FINAL OPTIMIZER
# =============================================================================

FINAL_OPTIMIZER_INPUTS = {

    "experiment_root": EXPERIMENT_ROOT,

    "stage_B": STAGES["B"],

    "stage_B2": STAGES["B2"],

    "stage_B3": STAGES["B3"],

    "stage_B34": STAGES["B3.4"],

    "stage_C1": STAGES["C1"],

    "stage_C2": STAGES["C2"],

    "stage_C3": STAGES["C3"],

    "stage_C4": STAGES["C4"],

    "stage_C42": STAGES["C4.2"],

    "stage_C43": STAGES["C4.3"],

    "C1_DATASET": C1_DATASET,

    "C2_OBJECTIVES": C2_OBJECTIVES,

    "C2_ALLOCATIONS": C2_ALLOCATIONS,

    "C2_PARETO": C2_PARETO,

    "C3_RESULTS": C3_RESULTS,

    "C42_TRIALS": C42_TRIALS,

    "C42_SUMMARY": C42_SUMMARY,

    "C42_ATTRIBUTION": C42_ATTRIBUTION,

    "C43_RESTORATIONS": C43_RESTORATIONS,

    "C43_LAYER_SUMMARY": C43_LAYER_SUMMARY,

    "C43_FORMAT_SUMMARY": C43_FORMAT_SUMMARY,

    "C43_EVIDENCE": C43_EVIDENCE,

    "MODEL_CHECKPOINT": MODEL_CHECKPOINT,

    "REFERENCE_FILE": REFERENCE_FILE
}

if preflight_passed:

    print("\n" + "=" * 80)

    print(
        "PREFLIGHT SUCCESSFUL — READY FOR FINAL OPTIMIZATION"
    )

    print("=" * 80)

    print(
        "\nAll required experiment stages and artifacts "
        "have passed the preflight checks."
    )

    print(
        "\nFinal optimizer inputs are available in:"
    )

    print("FINAL_OPTIMIZER_INPUTS")

else:

    print("\n" + "=" * 80)

    print(
        "PREFLIGHT INCOMPLETE — DO NOT START FINAL OPTIMIZER"
    )

    print("=" * 80)

    print(
        "Resolve the failed checks before optimization."
    )

print("\nJFAMP PREFLIGHT COMPLETE.")
print("=" * 80)


In [ ]:
from pathlib import Path
import os

INPUT_ROOT = Path("/kaggle/input")
TARGET = "revised_jfamp_fault_aware"

EXPECTED = [
    "C1_layer_format_optimizer_dataset.csv",
    "C2_candidate_layer_allocations.csv",
    "C3_clean_candidate_results.csv",
    "C4_2_fault_trials.csv",
    "C4_3_individual_bit_restorations.csv",
    "C4_3_final_evidence_summary.json",
]

print("=" * 100)
print("JFAMP — KAGGLE INPUT DIRECTORY DIAGNOSTIC")
print("=" * 100)

candidates = sorted(
    p for p in INPUT_ROOT.rglob(TARGET)
    if p.is_dir()
)

print(f"\nFound {len(candidates)} matching experiment directories.")

for idx, root in enumerate(candidates, 1):
    print("\n" + "=" * 100)
    print(f"CANDIDATE {idx}: {root}")
    print("=" * 100)

    all_files = sorted(p for p in root.rglob("*") if p.is_file())

    print(f"Total files: {len(all_files)}")

    print("\nDIRECTORY STRUCTURE:")
    directories = sorted(p for p in root.rglob("*") if p.is_dir())

    for directory in directories[:100]:
        print("DIR :", directory.relative_to(root))

    if len(directories) > 100:
        print(f"... {len(directories) - 100} additional directories")

    print("\nEXPECTED FILE SEARCH:")
    for filename in EXPECTED:
        matches = [p for p in all_files if p.name == filename]

        if matches:
            for match in matches:
                print(f"[FOUND]   {match}")
        else:
            print(f"[MISSING] {filename}")

    print("\nALL CSV / JSON FILES:")
    relevant = [
        p for p in all_files
        if p.suffix.lower() in (".csv", ".json")
    ]

    for p in relevant[:150]:
        print(p.relative_to(root))

    if len(relevant) > 150:
        print(f"... {len(relevant) - 150} additional CSV/JSON files")

print("\n" + "=" * 100)
print("OTHER JFAMP-RELATED INPUT DIRECTORIES")
print("=" * 100)

for p in sorted(INPUT_ROOT.iterdir()):
    if p.is_dir():
        print(p)

print("\nDIAGNOSTIC COMPLETE — NO FILES MODIFIED.")

In [ ]:
from pathlib import Path
import pandas as pd
import json
import hashlib

ROOT = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100-5/"
    "JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware"
)

STAGES = {
    "C1": {
        "dir": "stage_C1_optimizer_dataset",
        "files": [
            "C1_layer_format_optimizer_dataset.csv",
            "C1_B3_full_model_benchmarks.csv",
            "C1_B3_fault_benchmark_summary.csv",
            "C1_B34_coverage_audit.csv",
            "C1_dataset_manifest.json",
        ],
    },
    "C2": {
        "dir": "stage_C2_multiobjective_optimizer",
        "files": [
            "C2_candidate_objectives.csv",
            "C2_candidate_layer_allocations.csv",
            "C2_pareto_frontier.csv",
            "C2_optimizer_manifest.json",
        ],
    },
    "C3": {
        "dir": "stage_C3_clean_full_model_validation",
        "files": [
            "C3_clean_candidate_results.csv",
            "C3_experiment_manifest.json",
        ],
    },
    "C4": {
        "dir": "stage_C4_fault_validation",
        "files": [
            "C4_fault_trials.csv",
            "C4_fault_summary.csv",
            "C4_experiment_manifest.json",
        ],
    },
    "C4.2": {
        "dir": "stage_C4_2_extended_fault_validation",
        "files": [
            "C4_2_fault_trials.csv",
            "C4_2_fault_summary.csv",
            "C4_2_catastrophic_fault_attribution.csv",
            "C4_2_experiment_manifest.json",
        ],
    },
    "C4.3": {
        "dir": "stage_C4_3_final_fault_attribution",
        "files": [
            "C4_3_catastrophic_trial_reproduction.csv",
            "C4_3_individual_bit_restorations.csv",
            "C4_3_trial_recovery_summary.csv",
            "C4_3_vulnerable_layer_summary.csv",
            "C4_3_vulnerable_format_summary.csv",
            "C4_3_final_evidence_summary.json",
            "C4_3_experiment_manifest.json",
        ],
    },
}

print("=" * 90)
print("JFAMP — CORRECTED FINAL OPTIMIZER PREFLIGHT")
print("=" * 90)
print("Selected source:", ROOT)

assert ROOT.is_dir(), f"Experiment root not found: {ROOT}"

checks = []

def check(label, condition, detail=""):
    checks.append((label, bool(condition), detail))
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {label}" + (f" — {detail}" if detail else ""))

# 1. Stage artifact audit
for stage, config in STAGES.items():
    stage_dir = ROOT / config["dir"]
    check(f"{stage} directory", stage_dir.is_dir(), str(stage_dir))

    if not stage_dir.is_dir():
        continue

    for filename in config["files"]:
        path = stage_dir / filename
        check(f"{stage}: {filename}", path.is_file())

# 2. Load and validate optimizer dataset
c1_dir = ROOT / STAGES["C1"]["dir"]
dataset_path = c1_dir / "C1_layer_format_optimizer_dataset.csv"

if dataset_path.is_file():
    df = pd.read_csv(dataset_path)
    check("C1 dataset row count", len(df) == 371, f"{len(df)} rows")

    if "layer_name" in df.columns:
        check(
            "C1 unique layers",
            df["layer_name"].nunique() == 53,
            f"{df['layer_name'].nunique()} layers"
        )

    if "format" in df.columns:
        check(
            "C1 datatype count",
            df["format"].nunique() == 7,
            f"{df['format'].nunique()} formats"
        )

    print("\nC1 columns:", df.columns.tolist())

# 3. Candidate allocation audit
c2_path = (
    ROOT / STAGES["C2"]["dir"] /
    "C2_candidate_layer_allocations.csv"
)

if c2_path.is_file():
    c2 = pd.read_csv(c2_path)
    check("C2 candidate allocation file nonempty", len(c2) > 0,
          f"{len(c2)} rows")

    if "candidate_id" in c2.columns:
        check(
            "CAND_01269 exists",
            "CAND_01269" in set(c2["candidate_id"].astype(str))
        )

    print("\nC2 columns:", c2.columns.tolist())

# 4. Clean full-model validation audit
c3_path = (
    ROOT / STAGES["C3"]["dir"] /
    "C3_clean_candidate_results.csv"
)

if c3_path.is_file():
    c3 = pd.read_csv(c3_path)
    check("C3 results nonempty", len(c3) > 0, f"{len(c3)} rows")
    print("\nC3 columns:", c3.columns.tolist())

# 5. Extended fault trial audit
c42_path = (
    ROOT / STAGES["C4.2"]["dir"] /
    "C4_2_fault_trials.csv"
)

if c42_path.is_file():
    c42 = pd.read_csv(c42_path)
    check("C4.2 trial count", len(c42) == 200, f"{len(c42)} rows")

    if "status" in c42.columns:
        successful = c42["status"].astype(str).str.lower().isin(
            ["success", "successful", "ok", "complete", "completed"]
        ).sum()
        check("C4.2 successful trials", successful == 200,
              f"{successful}/200")

    print("\nC4.2 columns:", c42.columns.tolist())

# 6. Final bit-restoration evidence audit
c43_dir = ROOT / STAGES["C4.3"]["dir"]
restoration_path = c43_dir / "C4_3_individual_bit_restorations.csv"
reproduction_path = c43_dir / "C4_3_catastrophic_trial_reproduction.csv"
evidence_path = c43_dir / "C4_3_final_evidence_summary.json"

if restoration_path.is_file():
    restorations = pd.read_csv(restoration_path)
    check(
        "C4.3 restoration records",
        len(restorations) == 57,
        f"{len(restorations)} rows"
    )
    print("\nC4.3 restoration columns:", restorations.columns.tolist())

if reproduction_path.is_file():
    reproductions = pd.read_csv(reproduction_path)
    check(
        "C4.3 catastrophic reproductions",
        len(reproductions) == 8,
        f"{len(reproductions)} rows"
    )

if evidence_path.is_file():
    with open(evidence_path) as f:
        evidence = json.load(f)

    print("\nC4.3 final evidence summary:")
    print(json.dumps(evidence, indent=2)[:6000])

# 7. Manifest integrity / JSON readability
print("\nMANIFEST VALIDATION")
for stage, config in STAGES.items():
    for filename in config["files"]:
        if filename.endswith(".json"):
            path = ROOT / config["dir"] / filename
            if path.is_file():
                try:
                    with open(path) as f:
                        json.load(f)
                    check(f"{stage}: valid JSON — {filename}", True)
                except Exception as e:
                    check(f"{stage}: valid JSON — {filename}", False, str(e))

# 8. Final report
print("\n" + "=" * 90)
print("PREFLIGHT SUMMARY")
print("=" * 90)

passed = sum(ok for _, ok, _ in checks)
failed = len(checks) - passed

print(f"Checks passed: {passed}")
print(f"Checks failed: {failed}")

if failed:
    print("\nFAILED CHECKS:")
    for label, ok, detail in checks:
        if not ok:
            print(" -", label, detail)
    raise RuntimeError(
        "Preflight did not pass. Resolve the listed issues before final optimization."
    )

FINAL_OPTIMIZER_ROOT = ROOT
FINAL_OPTIMIZER_C1 = c1_dir
FINAL_OPTIMIZER_C2 = ROOT / STAGES["C2"]["dir"]
FINAL_OPTIMIZER_C3 = ROOT / STAGES["C3"]["dir"]
FINAL_OPTIMIZER_C42 = ROOT / STAGES["C4.2"]["dir"]
FINAL_OPTIMIZER_C43 = c43_dir

print("\nALL PREFLIGHT CHECKS PASSED.")
print("Final optimizer input root:", FINAL_OPTIMIZER_ROOT)
print("No files modified.")

In [ ]:
from pathlib import Path
import json, hashlib, time
import numpy as np
import pandas as pd

# ============================================================
# JFAMP STAGE C5 — FINAL RISK-AWARE MULTIOBJECTIVE OPTIMIZER
# ============================================================

ROOT = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100-5/"
    "JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware"
)

C1_DIR = ROOT / "stage_C1_optimizer_dataset"
C2_DIR = ROOT / "stage_C2_multiobjective_optimizer"
C3_DIR = ROOT / "stage_C3_clean_full_model_validation"
C43_DIR = ROOT / "stage_C4_3_final_fault_attribution"

OUT = Path(
    "/kaggle/working/JFAMP_MobileNetV2/experiments/"
    "revised_jfamp_fault_aware/stage_C5_final_risk_aware_optimizer"
)
OUT.mkdir(parents=True, exist_ok=True)

START = time.time()
VALIDATED_REFERENCE = "CAND_01269"
FP32_WEIGHT_BYTES = 9_271_040
STORAGE_BUDGET = int(0.60 * FP32_WEIGHT_BYTES)
BASELINE_ACCURACY = 68.06
MIN_CLEAN_ACCURACY = 67.06

print("=" * 90)
print("JFAMP STAGE C5 — FINAL RISK-AWARE OPTIMIZATION")
print("=" * 90)

# ------------------------------------------------------------
# 1. Load and validate source data
# ------------------------------------------------------------

c1 = pd.read_csv(C1_DIR / "C1_layer_format_optimizer_dataset.csv")
c2 = pd.read_csv(C2_DIR / "C2_candidate_layer_allocations.csv")
c3 = pd.read_csv(C3_DIR / "C3_clean_candidate_results.csv")
c43 = pd.read_csv(C43_DIR / "C4_3_individual_bit_restorations.csv")

required_c1 = {
    "layer", "format", "payload_bytes", "clean_drop_pp",
    "mean_degradation_1bit_pp", "mean_degradation_8bit_pp",
    "worst_observed_1bit_pp", "worst_observed_8bit_pp",
    "b34_exponent_tests", "b34_exponent_p95_pp",
    "b34_exponent_worst_pp"
}
required_c2 = {"candidate_id", "layer", "format", "payload_bytes"}
required_c3 = {"candidate_id", "accuracy_percent", "clean_feasible"}
required_c43 = {
    "layer", "format", "accuracy_recovered_pp",
    "exact_clean_predictions_restored", "status"
}

for name, frame, required in [
    ("C1", c1, required_c1),
    ("C2", c2, required_c2),
    ("C3", c3, required_c3),
    ("C4.3", c43, required_c43),
]:
    missing = required - set(frame.columns)
    if missing:
        raise RuntimeError(f"{name} is missing columns: {sorted(missing)}")

if len(c1) != 371:
    raise RuntimeError(f"Expected 371 C1 records, found {len(c1)}")

if c1.duplicated(["layer", "format"]).any():
    raise RuntimeError("C1 contains duplicate layer-format records.")

if c2.duplicated(["candidate_id", "layer"]).any():
    raise RuntimeError("C2 contains duplicate candidate-layer records.")

print(f"C1 records: {len(c1)}")
print(f"C2 allocation records: {len(c2)}")
print(f"C2 candidates: {c2['candidate_id'].nunique()}")
print(f"C3 validated candidates: {len(c3)}")
print(f"C4.3 restoration records: {len(c43)}")

# ------------------------------------------------------------
# 2. Construct the C4.3 empirical critical-bit evidence table
# ------------------------------------------------------------

# Restoration records are selected catastrophic-trial evidence.
# They are not estimates of population-wide fault probability.

c43["accuracy_recovered_pp"] = pd.to_numeric(
    c43["accuracy_recovered_pp"], errors="coerce"
).fillna(0.0)

c43["exact_clean_predictions_restored"] = (
    c43["exact_clean_predictions_restored"]
    .astype(str).str.lower().isin(["true", "1", "yes"])
)

c43_valid = c43[
    c43["status"].astype(str).str.lower().isin(
        ["success", "successful", "ok", "complete", "completed"]
    )
].copy()

critical = (
    c43_valid.groupby(["layer", "format"], as_index=False)
    .agg(
        c43_restoration_tests=("accuracy_recovered_pp", "size"),
        c43_max_recovery_pp=("accuracy_recovered_pp", "max"),
        c43_mean_recovery_pp=("accuracy_recovered_pp", "mean"),
        c43_exact_restorations=(
            "exact_clean_predictions_restored", "sum"
        )
    )
)

print(f"C4.3 layer-format combinations with direct evidence: {len(critical)}")

# ------------------------------------------------------------
# 3. Join per-layer metrics to every C2 candidate allocation
# ------------------------------------------------------------

metrics = c1.merge(
    critical,
    on=["layer", "format"],
    how="left",
    validate="one_to_one"
)

metrics["c43_evidence_observed"] = metrics[
    "c43_restoration_tests"
].notna()

metrics["c43_restoration_tests"] = (
    metrics["c43_restoration_tests"].fillna(0)
)
metrics["c43_max_recovery_pp"] = (
    metrics["c43_max_recovery_pp"].fillna(0.0)
)
metrics["c43_mean_recovery_pp"] = (
    metrics["c43_mean_recovery_pp"].fillna(0.0)
)
metrics["c43_exact_restorations"] = (
    metrics["c43_exact_restorations"].fillna(0)
)

alloc = c2.merge(
    metrics,
    on=["layer", "format"],
    how="left",
    suffixes=("_c2", ""),
    validate="many_to_one"
)

if alloc["mean_degradation_8bit_pp"].isna().any():
    raise RuntimeError("Some candidate assignments lack C1 metrics.")

alloc["payload_bytes_effective"] = pd.to_numeric(
    alloc["payload_bytes"], errors="coerce"
)

if alloc["payload_bytes_effective"].isna().any():
    raise RuntimeError("Missing payload sizes in candidate allocations.")

# ------------------------------------------------------------
# 4. Aggregate each complete allocation
# ------------------------------------------------------------

records = []

for candidate_id, group in alloc.groupby("candidate_id", sort=True):

    if group["layer"].nunique() != 53 or len(group) != 53:
        continue

    storage = int(group["payload_bytes_effective"].sum())

    clean_proxy = float(group["clean_drop_pp"].sum())

    expected_fault_proxy = float(
        (
            group["mean_degradation_1bit_pp"].astype(float)
            + group["mean_degradation_8bit_pp"].astype(float)
        ).mean() / 2.0
    )

    worst_layer_proxy = float(
        np.maximum(
            group["worst_observed_1bit_pp"].astype(float),
            group["worst_observed_8bit_pp"].astype(float)
        ).max()
    )

    exponent_measured = pd.to_numeric(
        group["b34_exponent_tests"], errors="coerce"
    ).fillna(0) > 0

    exponent_coverage = float(exponent_measured.mean())

    exponent_p95 = pd.to_numeric(
        group.loc[exponent_measured, "b34_exponent_p95_pp"],
        errors="coerce"
    )

    exponent_worst = pd.to_numeric(
        group.loc[exponent_measured, "b34_exponent_worst_pp"],
        errors="coerce"
    )

    # Missing B3.4 evidence remains missing; it is not treated as zero.
    exponent_p95_proxy = (
        float(exponent_p95.mean())
        if exponent_p95.notna().any() else np.nan
    )

    exponent_worst_proxy = (
        float(exponent_worst.max())
        if exponent_worst.notna().any() else np.nan
    )

    c43_observed = group["c43_evidence_observed"].astype(bool)
    c43_coverage = float(c43_observed.mean())

    c43_max_recovery = float(
        group["c43_max_recovery_pp"].max()
    )

    c43_exact_count = int(
        group["c43_exact_restorations"].sum()
    )

    records.append({
        "candidate_id": candidate_id,
        "layer_count": 53,
        "storage_bytes": storage,
        "storage_fraction_of_fp32": storage / FP32_WEIGHT_BYTES,
        "storage_feasible": storage <= STORAGE_BUDGET,
        "clean_drop_proxy_sum_pp": clean_proxy,
        "expected_fault_proxy_pp": expected_fault_proxy,
        "worst_layer_fault_proxy_pp": worst_layer_proxy,
        "b34_exponent_coverage_fraction": exponent_coverage,
        "b34_exponent_p95_proxy_pp": exponent_p95_proxy,
        "b34_exponent_worst_proxy_pp": exponent_worst_proxy,
        "c43_direct_evidence_coverage_fraction": c43_coverage,
        "c43_max_restoration_recovery_pp": c43_max_recovery,
        "c43_exact_restoration_count": c43_exact_count,
    })

summary = pd.DataFrame(records)

if summary.empty:
    raise RuntimeError("No complete 53-layer candidates were found.")

# ------------------------------------------------------------
# 5. Attach experimentally validated clean-model outcomes
# ------------------------------------------------------------

c3_small = c3[
    ["candidate_id", "accuracy_percent", "clean_feasible", "status"]
].copy()

c3_small = c3_small.rename(columns={
    "accuracy_percent": "measured_clean_accuracy_percent",
    "clean_feasible": "measured_clean_feasible",
    "status": "c3_validation_status"
})

summary = summary.merge(
    c3_small,
    on="candidate_id",
    how="left",
    validate="one_to_one"
)

summary["end_to_end_clean_validated"] = (
    summary["measured_clean_feasible"].fillna(False).astype(bool)
)

summary["clean_accuracy_constraint_known"] = (
    summary["measured_clean_accuracy_percent"].notna()
)

# CAND_01269 must remain identifiable as the validated reference.
if VALIDATED_REFERENCE not in set(summary["candidate_id"]):
    raise RuntimeError("Validated reference CAND_01269 is missing.")

# ------------------------------------------------------------
# 6. Build a Pareto frontier
# ------------------------------------------------------------

# All objectives are minimized.
# For the exponent and C4.3 metrics, candidates without coverage
# are not assigned zero risk. Their uncertainty is tracked separately.

objectives = [
    "storage_bytes",
    "clean_drop_proxy_sum_pp",
    "expected_fault_proxy_pp",
    "worst_layer_fault_proxy_pp",
]

# Only compare candidates with comparable measured exponent coverage
# and C4.3 coverage by keeping coverage as an additional minimized
# uncertainty objective (1 - coverage).
summary["b34_exponent_uncertainty"] = (
    1.0 - summary["b34_exponent_coverage_fraction"]
)
summary["c43_evidence_uncertainty"] = (
    1.0 - summary["c43_direct_evidence_coverage_fraction"]
)

objectives += [
    "b34_exponent_uncertainty",
    "c43_evidence_uncertainty",
]

# Add numerical exponent vulnerability only where measured.
# NaNs are conservatively replaced by the worst observed measured value
# for Pareto screening, while the original coverage is retained.
for col in [
    "b34_exponent_p95_proxy_pp",
    "b34_exponent_worst_proxy_pp"
]:
    finite = summary[col].dropna()
    fallback = float(finite.max()) if len(finite) else 0.0
    summary[col + "_screening"] = summary[col].fillna(fallback)
    objectives.append(col + "_screening")

# C4.3 restoration recovery is direct evidence only for observed
# layer-format pairs. Coverage remains an independent objective.
objectives.append("c43_max_restoration_recovery_pp")

# Drop candidates outside the hard storage budget from the frontier.
eligible = summary[summary["storage_feasible"]].copy()

X = eligible[objectives].to_numpy(dtype=float)
n = len(eligible)
is_pareto = np.ones(n, dtype=bool)

for i in range(n):
    if not is_pareto[i]:
        continue

    dominates_i = np.all(X <= X[i], axis=1) & np.any(X < X[i], axis=1)
    dominates_i[i] = False

    if dominates_i.any():
        is_pareto[i] = False

eligible["is_final_pareto"] = is_pareto
pareto_ids = set(
    eligible.loc[eligible["is_final_pareto"], "candidate_id"]
)

summary["is_final_pareto"] = summary["candidate_id"].isin(pareto_ids)

# ------------------------------------------------------------
# 7. Produce a transparent balanced ranking
# ------------------------------------------------------------

# Rank each objective independently. Lower rank is better.
# Equal weighting is used only as a shortlist heuristic, not as
# a physical probability model or a claim of global optimality.

rank_columns = [
    "storage_bytes",
    "clean_drop_proxy_sum_pp",
    "expected_fault_proxy_pp",
    "worst_layer_fault_proxy_pp",
    "b34_exponent_uncertainty",
    "c43_evidence_uncertainty",
    "b34_exponent_p95_proxy_pp_screening",
    "b34_exponent_worst_proxy_pp_screening",
    "c43_max_restoration_recovery_pp",
]

for col in rank_columns:
    summary[col + "_rank"] = summary[col].rank(
        method="average", ascending=True, na_option="bottom"
    )

summary["balanced_rank_score"] = summary[
    [col + "_rank" for col in rank_columns]
].mean(axis=1)

summary["balanced_rank"] = summary[
    "balanced_rank_score"
].rank(method="min", ascending=True).astype(int)

summary["reference_candidate"] = (
    summary["candidate_id"] == VALIDATED_REFERENCE
)

# Candidates with measured C3 clean results are clearly separated.
summary["validation_class"] = np.select(
    [
        summary["end_to_end_clean_validated"],
        summary["clean_accuracy_constraint_known"],
    ],
    [
        "END_TO_END_CLEAN_FEASIBLE",
        "END_TO_END_CLEAN_INFEASIBLE",
    ],
    default="NOT_YET_END_TO_END_VALIDATED"
)

# ------------------------------------------------------------
# 8. Build final shortlist and allocation exports
# ------------------------------------------------------------

shortlist = (
    summary[summary["storage_feasible"]]
    .sort_values(
        ["balanced_rank_score", "expected_fault_proxy_pp"],
        ascending=[True, True]
    )
    .head(25)
    .copy()
)

# Always include the validated reference, even if it is not in top 25.
if VALIDATED_REFERENCE not in set(shortlist["candidate_id"]):
    shortlist = pd.concat([
        shortlist,
        summary[summary["candidate_id"] == VALIDATED_REFERENCE]
    ], ignore_index=True).drop_duplicates("candidate_id")

pareto = summary[summary["is_final_pareto"]].copy()

# Candidate layer assignments for the Pareto frontier and shortlist.
selected_ids = set(shortlist["candidate_id"]) | pareto_ids | {
    VALIDATED_REFERENCE
}

selected_allocations = alloc[
    alloc["candidate_id"].isin(selected_ids)
].copy()

# ------------------------------------------------------------
# 9. Create an audit of CAND_01269
# ------------------------------------------------------------

reference_summary = summary[
    summary["candidate_id"] == VALIDATED_REFERENCE
].iloc[0]

reference_allocation = selected_allocations[
    selected_allocations["candidate_id"] == VALIDATED_REFERENCE
].copy()

reference_allocation = reference_allocation.sort_values("layer_index")

# ------------------------------------------------------------
# 10. Save outputs
# ------------------------------------------------------------

summary_path = OUT / "C5_all_candidate_risk_aware_scores.csv"
pareto_path = OUT / "C5_final_pareto_frontier.csv"
shortlist_path = OUT / "C5_final_candidate_shortlist.csv"
alloc_path = OUT / "C5_selected_candidate_layer_allocations.csv"
reference_path = OUT / "C5_validated_reference_allocation.csv"
evidence_path = OUT / "C5_C43_critical_bit_evidence.csv"
manifest_path = OUT / "C5_final_optimizer_manifest.json"

summary.to_csv(summary_path, index=False)
pareto.to_csv(pareto_path, index=False)
shortlist.to_csv(shortlist_path, index=False)
selected_allocations.to_csv(alloc_path, index=False)
reference_allocation.to_csv(reference_path, index=False)
critical.to_csv(evidence_path, index=False)

manifest = {
    "stage": "C5",
    "name": "Final risk-aware multiobjective optimizer",
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "input_root": str(ROOT),
    "output_directory": str(OUT),
    "input_records": {
        "C1_layer_format_records": int(len(c1)),
        "C2_allocation_records": int(len(c2)),
        "C2_complete_candidates": int(summary.shape[0]),
        "C3_end_to_end_validated_candidates": int(len(c3)),
        "C4_3_restoration_records": int(len(c43)),
        "C4_3_layer_format_evidence_records": int(len(critical)),
    },
    "constraints": {
        "fp32_quantizable_weight_bytes": FP32_WEIGHT_BYTES,
        "storage_budget_fraction": 0.60,
        "storage_budget_bytes": STORAGE_BUDGET,
        "fp32_baseline_accuracy_percent": BASELINE_ACCURACY,
        "minimum_clean_accuracy_percent": MIN_CLEAN_ACCURACY,
    },
    "validated_reference": {
        "candidate_id": VALIDATED_REFERENCE,
        "measured_clean_accuracy_percent": float(
            reference_summary["measured_clean_accuracy_percent"]
        ),
        "storage_bytes": int(reference_summary["storage_bytes"]),
        "storage_fraction": float(
            reference_summary["storage_fraction_of_fp32"]
        ),
        "end_to_end_clean_validated": bool(
            reference_summary["end_to_end_clean_validated"]
        ),
    },
    "optimization_method": {
        "candidate_source": "Existing C2 candidate allocations",
        "pareto_method": "Nondominated sorting across listed objectives",
        "balanced_shortlist": "Equal-weight average of objective ranks",
        "direct_fault_evidence": "C4.3 individual bit restoration records",
        "missing_measurements": (
            "Retained as missing; coverage and uncertainty explicitly tracked"
        ),
    },
    "interpretation_limits": [
        "C1 layer-wise clean and fault metrics are screening proxies.",
        "Additive layer metrics do not establish full-model clean accuracy.",
        "C4.3 evidence comes from selected catastrophic trials, not a random population sample.",
        "Unobserved C4.3 layer-format combinations are not considered safe.",
        "Only CAND_01269 among the six C3 candidates passed the measured clean-accuracy constraint.",
        "New candidates require full-model clean validation and fault validation.",
        "No ECC, parity, redundancy, or protection mechanism is included.",
    ],
    "outputs": {
        "all_candidate_scores": str(summary_path),
        "pareto_frontier": str(pareto_path),
        "candidate_shortlist": str(shortlist_path),
        "selected_layer_allocations": str(alloc_path),
        "validated_reference_allocation": str(reference_path),
        "critical_bit_evidence": str(evidence_path),
    },
    "runtime_seconds": round(time.time() - START, 2),
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# ------------------------------------------------------------
# 11. Print final results
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("C5 OPTIMIZATION RESULTS")
print("=" * 90)

print(f"Complete candidates evaluated: {len(summary)}")
print(f"Storage-feasible candidates: {int(summary['storage_feasible'].sum())}")
print(f"Pareto candidates: {len(pareto)}")
print(f"Shortlist candidates: {len(shortlist)}")

print("\nVALIDATED REFERENCE — CAND_01269")
for key in [
    "measured_clean_accuracy_percent",
    "storage_bytes",
    "storage_fraction_of_fp32",
    "expected_fault_proxy_pp",
    "worst_layer_fault_proxy_pp",
    "b34_exponent_coverage_fraction",
    "c43_direct_evidence_coverage_fraction",
]:
    print(f"{key}: {reference_summary[key]}")

print("\nTOP 15 RISK-AWARE CANDIDATES")
display_columns = [
    "candidate_id",
    "balanced_rank",
    "storage_bytes",
    "storage_fraction_of_fp32",
    "clean_drop_proxy_sum_pp",
    "expected_fault_proxy_pp",
    "worst_layer_fault_proxy_pp",
    "b34_exponent_coverage_fraction",
    "c43_direct_evidence_coverage_fraction",
    "c43_max_restoration_recovery_pp",
    "validation_class",
    "reference_candidate",
]

print(
    shortlist.sort_values("balanced_rank_score")[
        display_columns
    ].head(15).to_string(index=False)
)

print("\nPARETO FRONTIER — FIRST 20")
print(
    pareto.sort_values("balanced_rank_score")[
        display_columns
    ].head(20).to_string(index=False)
)

print("\nOUTPUT FILES")
for p in sorted(OUT.iterdir()):
    print(f"{p.name:55s} {p.stat().st_size:>12,} bytes")

print("\n" + "=" * 90)
print("C5 COMPLETE")
print(f"Runtime: {time.time() - START:.2f} seconds")
print("Reference allocation preserved:", VALIDATED_REFERENCE)
print("Output directory:", OUT)
print("=" * 90)

In [ ]:
from pathlib import Path
import os, json, copy, hashlib, ast, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

# ============================================================
# JFAMP C6 — FULL-MODEL CLEAN VALIDATION
# ============================================================

INPUT = Path("/kaggle/input")
ROOT = INPUT / "notebooks/crahulnaidu/mobile-net-100-5/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware"
C2_DIR = ROOT / "stage_C2_multiobjective_optimizer"

OUT = Path(
    "/kaggle/working/JFAMP_MobileNetV2/experiments/"
    "revised_jfamp_fault_aware/stage_C6_clean_full_model_validation"
)
OUT.mkdir(parents=True, exist_ok=True)

CANDIDATES = [
    "CAND_00361", "CAND_00967", "CAND_00853",
    "CAND_01103", "CAND_00778", "CAND_01218",
    "CAND_01269"
]

BASELINE = 68.06
MIN_ACCURACY = 67.06
MAX_STORAGE_FRACTION = 0.60
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_SHA256 = (
    "07c3c343451dcef593efc9172b9e07194b72f7c8c635b115f03104d7a8b2ff0a"
)

REFERENCE_NAME = "fp32_reference_predictions.npz"
CHECKPOINT_NAME = "mobilenetv2_cifar100_best.pth"

print("=" * 85)
print("JFAMP C6 — FULL-MODEL CLEAN VALIDATION")
print("=" * 85)
print("Device:", DEVICE)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# ------------------------------------------------------------
# 1. Locate checkpoint and canonical reference
# ------------------------------------------------------------

checkpoints = list(INPUT.rglob(CHECKPOINT_NAME))
checkpoints = [
    p for p in checkpoints
    if p.is_file() and sha256(p) == CHECKPOINT_SHA256
]
if not checkpoints:
    raise RuntimeError("Canonical checkpoint not found or SHA256 mismatch.")

checkpoint_path = checkpoints[0]
print("Checkpoint:", checkpoint_path)

references = list(INPUT.rglob(REFERENCE_NAME))
if not references:
    raise RuntimeError("Canonical FP32 reference predictions not found.")

valid_references = []
for p in references:
    try:
        z = np.load(p, allow_pickle=False)
        if (
            z["logits"].shape == (5000, 100)
            and z["predictions"].shape == (5000,)
            and z["labels"].shape == (5000,)
            and z["validation_indices"].shape == (5000,)
        ):
            valid_references.append(p)
    except Exception:
        pass

if not valid_references:
    raise RuntimeError("No valid 5,000-sample canonical reference found.")

reference_path = valid_references[0]
ref = np.load(reference_path, allow_pickle=False)
validation_indices = ref["validation_indices"].astype(np.int64)
reference_predictions = ref["predictions"].astype(np.int64)
reference_labels = ref["labels"].astype(np.int64)
reference_logits = ref["logits"].astype(np.float32)

print("Canonical reference:", reference_path)

# ------------------------------------------------------------
# 2. Locate CIFAR-100 data
# ------------------------------------------------------------

train_dirs = list(INPUT.rglob("cifar-100-python/train"))
train_dirs = [p for p in train_dirs if p.is_file()]

if not train_dirs:
    raise RuntimeError("CIFAR-100 extracted training file not found.")

data_root = train_dirs[0].parent.parent

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

full_dataset = datasets.CIFAR100(
    root=str(data_root),
    train=True,
    download=False,
    transform=eval_transform
)

if len(full_dataset) != 50000:
    raise RuntimeError(f"Unexpected CIFAR-100 train size: {len(full_dataset)}")

val_dataset = Subset(full_dataset, validation_indices.tolist())
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

# ------------------------------------------------------------
# 3. Locate exact validated codec implementations
# ------------------------------------------------------------

required_functions = {
    "encode_weights", "decode_weights",
    "encode_e4m11", "decode_e4m11",
    "pack_int4", "unpack_int4"
}

codec_namespace = {
    "np": np,
    "torch": torch,
    "nn": nn,
}

found_definitions = set()
source_files = list(INPUT.rglob("*.py")) + list(INPUT.rglob("*.ipynb"))

for source_path in source_files:
    try:
        if source_path.suffix == ".py":
            source_text = source_path.read_text(errors="ignore")
            cells = [source_text]
        else:
            notebook = json.loads(source_path.read_text(errors="ignore"))
            cells = [
                "".join(cell.get("source", []))
                for cell in notebook.get("cells", [])
                if cell.get("cell_type") == "code"
            ]

        for source in cells:
            try:
                tree = ast.parse(source)
            except Exception:
                continue

            definitions = {
                node.name for node in tree.body
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            }

            relevant = definitions & required_functions
            if not relevant:
                continue

            # Execute the original source cell, preserving its exact codec code.
            exec(compile(source, str(source_path), "exec"), codec_namespace)
            found_definitions.update(relevant)

    except Exception:
        continue

missing = required_functions - found_definitions
if missing:
    raise RuntimeError(
        "Could not load all exact validated codec definitions: "
        + ", ".join(sorted(missing))
        + ". Mount the original unified Stage 2 notebook/source containing "
          "the validated codec implementations."
    )

encode_weights = codec_namespace["encode_weights"]
decode_weights = codec_namespace["decode_weights"]

print("Exact codec definitions loaded:", sorted(found_definitions))

# ------------------------------------------------------------
# 4. Load model checkpoint and reproduce FP32 reference
# ------------------------------------------------------------

checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)

model = models.mobilenet_v2(weights=None, num_classes=100)
model.features[0][0].stride = (1, 1)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model = model.to(DEVICE).eval()

def evaluate(model):
    logits_all = []
    labels_all = []

    model.eval()
    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            logits = model(images)
            logits_all.append(logits.float().cpu().numpy())
            labels_all.append(labels.numpy())

    logits = np.concatenate(logits_all)
    labels = np.concatenate(labels_all)
    predictions = logits.argmax(axis=1)
    accuracy = 100.0 * np.mean(predictions == labels)

    return {
        "logits": logits,
        "labels": labels,
        "predictions": predictions,
        "accuracy": float(accuracy),
        "correct_count": int(np.sum(predictions == labels))
    }

baseline_result = evaluate(model)

if not np.array_equal(baseline_result["labels"], reference_labels):
    raise RuntimeError("Validation labels differ from canonical reference.")

if not np.array_equal(
    baseline_result["predictions"], reference_predictions
):
    raise RuntimeError("FP32 predictions differ from canonical reference.")

if not np.allclose(
    baseline_result["logits"], reference_logits,
    rtol=0, atol=1e-6
):
    raise RuntimeError("FP32 logits differ from canonical reference.")

print(f"FP32 reproduced accuracy: {baseline_result['accuracy']:.4f}%")
print("FP32 reference reproduction: EXACT")

# ------------------------------------------------------------
# 5. Load candidate allocations
# ------------------------------------------------------------

allocation_path = C2_DIR / "C2_candidate_layer_allocations.csv"
allocations = pd.read_csv(allocation_path)

allocations = allocations[
    allocations["candidate_id"].isin(CANDIDATES)
].copy()

if set(allocations["candidate_id"].unique()) != set(CANDIDATES):
    raise RuntimeError("One or more requested candidates are missing.")

if allocations.groupby("candidate_id")["layer"].nunique().min() != 53:
    raise RuntimeError("A candidate does not contain all 53 layers.")

# ------------------------------------------------------------
# 6. Apply exact serialized codec to each candidate
# ------------------------------------------------------------

named_modules = dict(model.named_modules())
named_parameters = dict(model.named_parameters())

results = []
allocation_outputs = []

for candidate_id in CANDIDATES:
    print("\n" + "-" * 75)
    print("Evaluating:", candidate_id)

    group = allocations[
        allocations["candidate_id"] == candidate_id
    ].copy()

    candidate_model = copy.deepcopy(model).cpu()
    storage_bytes = 0
    candidate_records = []

    for row in group.itertuples(index=False):
        layer = str(row.layer)
        fmt = str(row.format)

        if layer not in named_modules:
            raise RuntimeError(f"Layer module not found: {layer}")

        module = named_modules[layer]
        if not hasattr(module, "weight") or module.weight is None:
            raise RuntimeError(f"Missing weight tensor for {layer}")

        weight = module.weight.detach().cpu().numpy().copy()

        payload, metadata = encode_weights(weight, fmt)
        reconstructed = decode_weights(payload, metadata, fmt)
        reconstructed = np.asarray(reconstructed)

        if reconstructed.shape != weight.shape:
            raise RuntimeError(
                f"Decoded shape mismatch for {layer} / {fmt}"
            )

        if not np.isfinite(reconstructed).all():
            raise RuntimeError(
                f"Non-finite reconstructed weights: {layer} / {fmt}"
            )

        actual_bytes = len(payload)
        expected_bytes = int(row.payload_bytes)

        if actual_bytes != expected_bytes:
            raise RuntimeError(
                f"Payload-size mismatch for {layer}/{fmt}: "
                f"{actual_bytes} != {expected_bytes}"
            )

        module.weight.data.copy_(
            torch.from_numpy(reconstructed).to(
                dtype=module.weight.dtype
            )
        )

        storage_bytes += actual_bytes

        candidate_records.append({
            "candidate_id": candidate_id,
            "layer": layer,
            "format": fmt,
            "payload_bytes": actual_bytes,
            "payload_sha256": hashlib.sha256(payload).hexdigest()
        })

    candidate_model = candidate_model.to(DEVICE).eval()
    result = evaluate(candidate_model)

    drop = BASELINE - result["accuracy"]
    storage_fraction = storage_bytes / 9_271_040

    clean_pass = result["accuracy"] >= MIN_ACCURACY
    storage_pass = storage_fraction <= MAX_STORAGE_FRACTION

    results.append({
        "candidate_id": candidate_id,
        "accuracy_percent": result["accuracy"],
        "accuracy_drop_vs_fp32_pp": drop,
        "loss": None,
        "correct_count": result["correct_count"],
        "sample_count": 5000,
        "storage_bytes": storage_bytes,
        "storage_fraction": storage_fraction,
        "clean_accuracy_pass": clean_pass,
        "storage_pass": storage_pass,
        "overall_clean_feasible": clean_pass and storage_pass,
        "status": "SUCCESS"
    })

    allocation_outputs.extend(candidate_records)

    print(f"Accuracy: {result['accuracy']:.4f}%")
    print(f"Accuracy drop: {drop:.4f} pp")
    print(f"Storage: {storage_bytes:,} bytes ({storage_fraction:.2%})")
    print("Clean accuracy:", "PASS" if clean_pass else "FAIL")

    del candidate_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# 7. Save results and manifest
# ------------------------------------------------------------

results_df = pd.DataFrame(results)
allocation_df = pd.DataFrame(allocation_outputs)

results_path = OUT / "C6_clean_candidate_results.csv"
allocation_out_path = OUT / "C6_candidate_payload_audit.csv"
manifest_path = OUT / "C6_experiment_manifest.json"

results_df.to_csv(results_path, index=False)
allocation_df.to_csv(allocation_out_path, index=False)

manifest = {
    "stage": "C6",
    "purpose": "Full-model clean validation of risk-aware candidates",
    "checkpoint_path": str(checkpoint_path),
    "checkpoint_sha256": sha256(checkpoint_path),
    "reference_path": str(reference_path),
    "reference_validation_samples": 5000,
    "fp32_reproduced_accuracy_percent": baseline_result["accuracy"],
    "minimum_clean_accuracy_percent": MIN_ACCURACY,
    "storage_budget_fraction": MAX_STORAGE_FRACTION,
    "candidates": CANDIDATES,
    "successful_candidates": len(results_df),
    "clean_feasible_candidates": results_df.loc[
        results_df["overall_clean_feasible"], "candidate_id"
    ].tolist(),
    "codec_source_files": sorted(found_definitions),
    "outputs": {
        "results": str(results_path),
        "payload_audit": str(allocation_out_path)
    },
    "runtime_seconds": round(time.time() - START, 2)
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("\n" + "=" * 85)
print("C6 FINAL CLEAN VALIDATION SUMMARY")
print("=" * 85)
print(results_df.to_string(index=False))
print("\nClean-feasible candidates:")
print(manifest["clean_feasible_candidates"])
print("\nSaved:", OUT)
print("C6 COMPLETE")

In [ ]:
from pathlib import Path
import json, hashlib, ast, time, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

# ============================================================
# JFAMP C6R — CORRECTED FULL-MODEL CLEAN VALIDATION
# ============================================================

INPUT = Path("/kaggle/input")
ROOT = INPUT / "notebooks/crahulnaidu/mobile-net-100-5/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware"

OUT = Path(
    "/kaggle/working/JFAMP_MobileNetV2/experiments/"
    "revised_jfamp_fault_aware/stage_C6_corrected_clean_validation"
)
OUT.mkdir(parents=True, exist_ok=True)

CANDIDATES = [
    "CAND_00361", "CAND_00967", "CAND_00853",
    "CAND_01103", "CAND_00778", "CAND_01218",
    "CAND_01269"
]

CHECKPOINT_NAME = "mobilenetv2_cifar100_best.pth"
REFERENCE_NAME = "fp32_reference_predictions.npz"
CHECKPOINT_SHA256 = "07c3c343451dcef593efc9172b9e07194b72f7c8c635b115f03104d7a8b2ff0a"

BASELINE_ACCURACY = 68.06
MIN_ACCURACY = 67.06
FP32_WEIGHT_BYTES = 9_271_040
MAX_STORAGE_FRACTION = 0.60
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

START = time.time()

print("=" * 85)
print("JFAMP C6R — CORRECTED CLEAN VALIDATION")
print("=" * 85)
print("Device:", DEVICE)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# ------------------------------------------------------------
# 1. Locate canonical checkpoint and reference
# ------------------------------------------------------------

checkpoint_matches = [
    p for p in INPUT.rglob(CHECKPOINT_NAME)
    if p.is_file() and sha256_file(p) == CHECKPOINT_SHA256
]

if not checkpoint_matches:
    raise RuntimeError("Canonical checkpoint missing or SHA256 mismatch.")

checkpoint_path = checkpoint_matches[0]

reference_matches = []
for p in INPUT.rglob(REFERENCE_NAME):
    try:
        z = np.load(p, allow_pickle=False)
        if (
            z["logits"].shape == (5000, 100)
            and z["predictions"].shape == (5000,)
            and z["labels"].shape == (5000,)
            and z["validation_indices"].shape == (5000,)
        ):
            reference_matches.append(p)
    except Exception:
        pass

if not reference_matches:
    raise RuntimeError("Canonical 5000-sample reference not found.")

reference_path = reference_matches[0]
ref = np.load(reference_path, allow_pickle=False)

indices = ref["validation_indices"].astype(np.int64)
ref_predictions = ref["predictions"].astype(np.int64)
ref_labels = ref["labels"].astype(np.int64)
ref_logits = ref["logits"].astype(np.float32)

print("Checkpoint:", checkpoint_path)
print("Reference:", reference_path)

# ------------------------------------------------------------
# 2. Reconstruct exact validation dataset
# ------------------------------------------------------------

train_files = [
    p for p in INPUT.rglob("cifar-100-python/train")
    if p.is_file()
]

if not train_files:
    raise RuntimeError("CIFAR-100 extracted training file not found.")

data_root = train_files[0].parent.parent

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

full_dataset = datasets.CIFAR100(
    root=str(data_root),
    train=True,
    download=False,
    transform=eval_transform
)

if len(full_dataset) != 50000:
    raise RuntimeError("Unexpected CIFAR-100 training dataset size.")

val_dataset = Subset(full_dataset, indices.tolist())

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

# ------------------------------------------------------------
# 3. Load exact codec definitions
# ------------------------------------------------------------

required_functions = {
    "encode_weights", "decode_weights",
    "encode_e4m11", "decode_e4m11",
    "pack_int4", "unpack_int4"
}

codec_namespace = {"np": np, "torch": torch, "nn": nn}
found = set()

source_files = list(INPUT.rglob("*.py")) + list(INPUT.rglob("*.ipynb"))

for source_path in source_files:
    try:
        if source_path.suffix == ".py":
            sources = [source_path.read_text(errors="ignore")]
        else:
            nb = json.loads(source_path.read_text(errors="ignore"))
            sources = [
                "".join(c.get("source", []))
                for c in nb.get("cells", [])
                if c.get("cell_type") == "code"
            ]

        for source in sources:
            try:
                tree = ast.parse(source)
            except Exception:
                continue

            defs = {
                n.name for n in tree.body
                if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
            }

            matched = defs & required_functions
            if matched:
                exec(compile(source, str(source_path), "exec"), codec_namespace)
                found.update(matched)

    except Exception:
        continue

missing = required_functions - found
if missing:
    raise RuntimeError(
        "Exact validated codec functions not found: " + str(sorted(missing))
    )

encode_weights = codec_namespace["encode_weights"]
decode_weights = codec_namespace["decode_weights"]

print("Codec functions verified:", sorted(found))

# ------------------------------------------------------------
# 4. Load immutable FP32 model
# ------------------------------------------------------------

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

base_model = models.mobilenet_v2(weights=None, num_classes=100)
base_model.features[0][0].stride = (1, 1)
base_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
base_model = base_model.to(DEVICE).eval()

def evaluate(model):
    model.eval()
    logits_list, labels_list = [], []
    loss_sum = 0.0
    count = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels_device = labels.to(DEVICE, non_blocking=True)

            logits = model(images)
            batch_loss = nn.functional.cross_entropy(
                logits, labels_device, reduction="sum"
            )

            loss_sum += float(batch_loss.item())
            count += len(labels)

            logits_list.append(logits.float().cpu().numpy())
            labels_list.append(labels.numpy())

    logits = np.concatenate(logits_list)
    labels = np.concatenate(labels_list)
    predictions = logits.argmax(axis=1)

    return {
        "logits": logits,
        "labels": labels,
        "predictions": predictions,
        "accuracy": 100.0 * float(np.mean(predictions == labels)),
        "correct_count": int(np.sum(predictions == labels)),
        "loss": loss_sum / count
    }

baseline = evaluate(base_model)

if not np.array_equal(baseline["labels"], ref_labels):
    raise RuntimeError("Validation labels do not match canonical reference.")

if not np.array_equal(baseline["predictions"], ref_predictions):
    raise RuntimeError("FP32 predictions do not match canonical reference.")

if not np.allclose(baseline["logits"], ref_logits, rtol=0, atol=1e-6):
    raise RuntimeError("FP32 logits do not match canonical reference.")

if abs(baseline["accuracy"] - BASELINE_ACCURACY) > 1e-6:
    raise RuntimeError("Unexpected FP32 baseline accuracy.")

print(f"FP32 accuracy: {baseline['accuracy']:.4f}%")
print(f"FP32 loss: {baseline['loss']:.8f}")
print("Canonical reference reproduction: EXACT")

# ------------------------------------------------------------
# 5. Load candidate allocations
# ------------------------------------------------------------

allocation_path = (
    ROOT / "stage_C2_multiobjective_optimizer" /
    "C2_candidate_layer_allocations.csv"
)

all_allocations = pd.read_csv(allocation_path)

allocations = all_allocations[
    all_allocations["candidate_id"].isin(CANDIDATES)
].copy()

if set(allocations["candidate_id"].unique()) != set(CANDIDATES):
    raise RuntimeError("Missing candidate allocation.")

for candidate_id, group in allocations.groupby("candidate_id"):
    if len(group) != 53 or group["layer"].nunique() != 53:
        raise RuntimeError(f"{candidate_id} does not have 53 unique layers.")

# ------------------------------------------------------------
# 6. Evaluate each independent candidate copy
# ------------------------------------------------------------

results = []
payload_audit = []

for candidate_id in CANDIDATES:
    print("\n" + "-" * 75)
    print("Candidate:", candidate_id)

    group = allocations[
        allocations["candidate_id"] == candidate_id
    ].copy()

    # CRITICAL FIX:
    # Create candidate first, then obtain modules FROM THAT COPY.
    candidate_model = copy.deepcopy(base_model).cpu().eval()
    candidate_modules = dict(candidate_model.named_modules())

    storage_bytes = 0
    layer_records = []

    for row in group.itertuples(index=False):
        layer = str(row.layer)
        fmt = str(row.format)

        if layer not in candidate_modules:
            raise RuntimeError(f"Module not found in candidate copy: {layer}")

        module = candidate_modules[layer]

        if not hasattr(module, "weight") or module.weight is None:
            raise RuntimeError(f"Missing weight: {layer}")

        original_weight = module.weight.detach().cpu().numpy().copy()

        payload, metadata = encode_weights(original_weight, fmt)
        decoded = np.asarray(
            decode_weights(payload, metadata, fmt)
        )

        if decoded.shape != original_weight.shape:
            raise RuntimeError(f"Shape mismatch: {layer}/{fmt}")

        if not np.isfinite(decoded).all():
            raise RuntimeError(f"Non-finite decoded weights: {layer}/{fmt}")

        actual_bytes = len(payload)
        expected_bytes = int(row.payload_bytes)

        if actual_bytes != expected_bytes:
            raise RuntimeError(
                f"Payload mismatch {layer}/{fmt}: "
                f"{actual_bytes} != {expected_bytes}"
            )

        # Apply quantized weights to the candidate's own module.
        module.weight.data.copy_(
            torch.from_numpy(decoded).to(dtype=module.weight.dtype)
        )

        storage_bytes += actual_bytes

        layer_records.append({
            "candidate_id": candidate_id,
            "layer": layer,
            "format": fmt,
            "payload_bytes": actual_bytes,
            "payload_sha256": hashlib.sha256(payload).hexdigest()
        })

    if len(layer_records) != 53:
        raise RuntimeError("Candidate did not quantize exactly 53 layers.")

    candidate_model = candidate_model.to(DEVICE).eval()

    result = evaluate(candidate_model)

    drop = baseline["accuracy"] - result["accuracy"]
    storage_fraction = storage_bytes / FP32_WEIGHT_BYTES

    clean_pass = result["accuracy"] >= MIN_ACCURACY
    storage_pass = storage_fraction <= MAX_STORAGE_FRACTION

    results.append({
        "candidate_id": candidate_id,
        "accuracy_percent": result["accuracy"],
        "accuracy_drop_vs_fp32_pp": drop,
        "loss": result["loss"],
        "correct_count": result["correct_count"],
        "sample_count": 5000,
        "storage_bytes": storage_bytes,
        "storage_fraction": storage_fraction,
        "clean_accuracy_pass": clean_pass,
        "storage_pass": storage_pass,
        "overall_clean_feasible": clean_pass and storage_pass,
        "status": "SUCCESS"
    })

    payload_audit.extend(layer_records)

    print(f"Accuracy: {result['accuracy']:.4f}%")
    print(f"Accuracy drop: {drop:.4f} pp")
    print(f"Loss: {result['loss']:.8f}")
    print(f"Storage: {storage_bytes:,} bytes ({storage_fraction:.2%})")
    print("Clean constraint:", "PASS" if clean_pass else "FAIL")

    del candidate_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# 7. Save corrected outputs
# ------------------------------------------------------------

results_df = pd.DataFrame(results)
audit_df = pd.DataFrame(payload_audit)

results_path = OUT / "C6R_clean_candidate_results.csv"
audit_path = OUT / "C6R_candidate_payload_audit.csv"
manifest_path = OUT / "C6R_experiment_manifest.json"

results_df.to_csv(results_path, index=False)
audit_df.to_csv(audit_path, index=False)

manifest = {
    "stage": "C6R",
    "purpose": "Corrected independent full-model clean validation",
    "correction": (
        "Candidate module lookup is built from each candidate model copy; "
        "the immutable FP32 base model is never modified."
    ),
    "checkpoint": str(checkpoint_path),
    "checkpoint_sha256": sha256_file(checkpoint_path),
    "reference": str(reference_path),
    "reference_samples": 5000,
    "baseline_accuracy_percent": baseline["accuracy"],
    "baseline_loss": baseline["loss"],
    "candidate_ids": CANDIDATES,
    "candidate_count": len(results_df),
    "clean_feasible_candidates": results_df.loc[
        results_df["overall_clean_feasible"], "candidate_id"
    ].tolist(),
    "storage_budget_fraction": MAX_STORAGE_FRACTION,
    "minimum_clean_accuracy_percent": MIN_ACCURACY,
    "codec_functions_loaded": sorted(found),
    "runtime_seconds": round(time.time() - START, 2),
    "outputs": {
        "results": str(results_path),
        "payload_audit": str(audit_path)
    }
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# ------------------------------------------------------------
# 8. Final report
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("C6R — CORRECTED CLEAN VALIDATION RESULTS")
print("=" * 85)

print(results_df.to_string(index=False))

print("\nClean-feasible candidates:")
print(manifest["clean_feasible_candidates"])

print("\nFiles saved:")
for p in [results_path, audit_path, manifest_path]:
    print(p, f"({p.stat().st_size:,} bytes)")

print("\nRuntime:", round(time.time() - START, 2), "seconds")
print("C6R COMPLETE")

In [ ]:
from pathlib import Path
import os, json, ast, time, copy, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

# ============================================================
# JFAMP C7 — EXTENDED FULL-MODEL GLOBAL FAULT VALIDATION
# ============================================================

INPUT = Path("/kaggle/input")
ROOT = INPUT / "notebooks/crahulnaidu/mobile-net-100-5/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware"

C2_DIR = ROOT / "stage_C2_multiobjective_optimizer"
C6_DIR = Path(
    "/kaggle/working/JFAMP_MobileNetV2/experiments/"
    "revised_jfamp_fault_aware/stage_C6_corrected_clean_validation"
)
C6_RESULT = C6_DIR / "C6R_clean_candidate_results.csv"

OUT = Path(
    "/kaggle/working/JFAMP_MobileNetV2/experiments/"
    "revised_jfamp_fault_aware/stage_C7_extended_fault_validation"
)
OUT.mkdir(parents=True, exist_ok=True)

CANDIDATES = ["CAND_00853", "CAND_00361", "CAND_01218"]
FAULT_COUNTS = [1, 8]
TRIALS_PER_SETTING = 100
BASE_SEED = 2026111600

CHECKPOINT_NAME = "mobilenetv2_cifar100_best.pth"
REFERENCE_NAME = "fp32_reference_predictions.npz"
CHECKPOINT_SHA256 = "07c3c343451dcef593efc9172b9e07194b72f7c8c635b115f03104d7a8b2ff0a"

BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FP32_WEIGHT_BYTES = 9_271_040

START = time.time()

print("=" * 90)
print("JFAMP C7 — EXTENDED GLOBAL FAULT VALIDATION")
print("=" * 90)
print("Device:", DEVICE)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# ------------------------------------------------------------
# 1. Verify C6R clean validation
# ------------------------------------------------------------

if not C6_RESULT.is_file():
    raise RuntimeError("C6R results missing. Complete corrected C6 first.")

c6 = pd.read_csv(C6_RESULT)

for candidate in CANDIDATES:
    row = c6[c6["candidate_id"] == candidate]
    if len(row) != 1:
        raise RuntimeError(f"Missing C6R result for {candidate}")
    if not bool(row.iloc[0]["overall_clean_feasible"]):
        raise RuntimeError(f"{candidate} is not clean-feasible.")

# ------------------------------------------------------------
# 2. Locate canonical model and reference
# ------------------------------------------------------------

# ------------------------------------------------------------
# 2. Locate the exact canonical model and reference
# ------------------------------------------------------------

checkpoints = [
    p for p in INPUT.rglob(CHECKPOINT_NAME)
    if p.is_file() and sha256_file(p) == CHECKPOINT_SHA256
]

if not checkpoints:
    raise RuntimeError("Canonical checkpoint missing or SHA mismatch.")

checkpoint_path = checkpoints[0]

REQUIRED_REFERENCE_KEYS = {
    "logits": (5000, 100),
    "predictions": (5000,),
    "labels": (5000,),
    "correct": (5000,),
    "validation_indices": (5000,)
}

valid_references = []
rejected_references = []

for p in INPUT.rglob(REFERENCE_NAME):
    try:
        z = np.load(p, allow_pickle=False)
        available = set(z.files)

        missing = set(REQUIRED_REFERENCE_KEYS) - available
        if missing:
            rejected_references.append(
                (str(p), f"Missing keys: {sorted(missing)}")
            )
            continue

        shape_errors = []
        for key, expected_shape in REQUIRED_REFERENCE_KEYS.items():
            if z[key].shape != expected_shape:
                shape_errors.append(
                    f"{key}: {z[key].shape}, expected {expected_shape}"
                )

        if shape_errors:
            rejected_references.append(
                (str(p), "; ".join(shape_errors))
            )
            continue

        valid_references.append(p)

    except Exception as e:
        rejected_references.append((str(p), str(e)))

if not valid_references:
    print("Rejected reference files:")
    for path, reason in rejected_references:
        print(path, "->", reason)

    raise RuntimeError(
        "No canonical reference archive with all required arrays was found."
    )

# Prefer the original canonical reference location when available.
preferred = [
    p for p in valid_references
    if "mobile-net-100-3" in str(p)
    and "stage_02_baseline" in str(p)
]

reference_path = preferred[0] if preferred else valid_references[0]

ref = np.load(reference_path, allow_pickle=False)

indices = ref["validation_indices"].astype(np.int64)
ref_predictions = ref["predictions"].astype(np.int64)
ref_labels = ref["labels"].astype(np.int64)
ref_logits = ref["logits"].astype(np.float32)
ref_correct = ref["correct"].astype(bool)

if len(np.unique(indices)) != 5000:
    raise RuntimeError("Canonical validation indices are not unique.")

if not np.array_equal(ref_correct, ref_predictions == ref_labels):
    raise RuntimeError("Reference correct mask is inconsistent.")

print("Canonical checkpoint:", checkpoint_path)
print("Checkpoint SHA256:", sha256_file(checkpoint_path))
print("Canonical reference:", reference_path)
print("Reference keys:", ref.files)
print("Validation samples:", len(indices))
print("Unique validation indices:", len(np.unique(indices)))
print("Reference accuracy:", 100 * ref_correct.mean())

# ------------------------------------------------------------
# 3. Reconstruct validation data
# ------------------------------------------------------------

train_files = [
    p for p in INPUT.rglob("cifar-100-python/train")
    if p.is_file()
]
if not train_files:
    raise RuntimeError("CIFAR-100 training file not found.")

data_root = train_files[0].parent.parent

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

full_dataset = datasets.CIFAR100(
    root=str(data_root),
    train=True,
    download=False,
    transform=eval_transform
)

val_loader = DataLoader(
    Subset(full_dataset, indices.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

# ------------------------------------------------------------
# 4. Load exact validated codecs
# ------------------------------------------------------------

required_functions = {
    "encode_weights", "decode_weights",
    "encode_e4m11", "decode_e4m11",
    "pack_int4", "unpack_int4"
}

namespace = {"np": np, "torch": torch, "nn": nn}
found = set()

for path in list(INPUT.rglob("*.py")) + list(INPUT.rglob("*.ipynb")):
    try:
        if path.suffix == ".py":
            sources = [path.read_text(errors="ignore")]
        else:
            nb = json.loads(path.read_text(errors="ignore"))
            sources = [
                "".join(c.get("source", []))
                for c in nb.get("cells", [])
                if c.get("cell_type") == "code"
            ]

        for source in sources:
            try:
                tree = ast.parse(source)
            except Exception:
                continue

            defs = {
                n.name for n in tree.body
                if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
            }
            matched = defs & required_functions

            if matched:
                exec(compile(source, str(path), "exec"), namespace)
                found.update(matched)

    except Exception:
        continue

missing = required_functions - found
if missing:
    raise RuntimeError(
        "Missing exact codec functions: " + str(sorted(missing))
    )

encode_weights = namespace["encode_weights"]
decode_weights = namespace["decode_weights"]

# ------------------------------------------------------------
# 5. Load immutable FP32 model and verify reference
# ------------------------------------------------------------

checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)

base_model = models.mobilenet_v2(weights=None, num_classes=100)
base_model.features[0][0].stride = (1, 1)
base_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
base_model = base_model.to(DEVICE).eval()

def evaluate(model):
    model.eval()
    predictions, labels = [], []
    with torch.inference_mode():
        for images, target in val_loader:
            logits = model(images.to(DEVICE, non_blocking=True))
            predictions.append(logits.argmax(1).cpu().numpy())
            labels.append(target.numpy())

    predictions = np.concatenate(predictions)
    labels = np.concatenate(labels)
    return {
        "predictions": predictions,
        "labels": labels,
        "accuracy": float(100 * np.mean(predictions == labels)),
        "correct_count": int(np.sum(predictions == labels))
    }

fp32 = evaluate(base_model)

if not np.array_equal(fp32["predictions"], ref_predictions):
    raise RuntimeError("FP32 reference predictions do not match.")
if not np.array_equal(fp32["labels"], ref_labels):
    raise RuntimeError("FP32 labels do not match.")

print(f"FP32 baseline reproduced: {fp32['accuracy']:.4f}%")

# ------------------------------------------------------------
# 6. Build each candidate's exact serialized payload
# ------------------------------------------------------------

allocations = pd.read_csv(
    C2_DIR / "C2_candidate_layer_allocations.csv"
)

def build_candidate(candidate_id):
    group = allocations[
        allocations["candidate_id"] == candidate_id
    ].copy()

    if len(group) != 53 or group["layer"].nunique() != 53:
        raise RuntimeError(f"Invalid allocation for {candidate_id}")

    candidate_model = copy.deepcopy(base_model).cpu().eval()
    modules = dict(candidate_model.named_modules())

    payloads = []
    records = []
    offset = 0
    total_bytes = 0

    for row in group.itertuples(index=False):
        layer = str(row.layer)
        fmt = str(row.format)

        if layer not in modules:
            raise RuntimeError(f"Missing module {layer}")

        module = modules[layer]
        original = module.weight.detach().cpu().numpy().copy()

        payload, metadata = encode_weights(original, fmt)
        payload = bytes(payload)

        if len(payload) != int(row.payload_bytes):
            raise RuntimeError(f"Payload size mismatch: {layer}/{fmt}")

        decoded = np.asarray(
            decode_weights(payload, metadata, fmt)
        )
        if not np.isfinite(decoded).all():
            raise RuntimeError(f"Non-finite clean decode: {layer}/{fmt}")

        module.weight.data.copy_(
            torch.from_numpy(decoded).to(dtype=module.weight.dtype)
        )

        payloads.append({
            "layer": layer,
            "format": fmt,
            "payload": payload,
            "metadata": metadata,
            "start_byte": offset,
            "end_byte": offset + len(payload)
        })

        records.append({
            "layer": layer,
            "format": fmt,
            "payload_bytes": len(payload),
            "payload_sha256": hashlib.sha256(payload).hexdigest()
        })

        offset += len(payload)
        total_bytes += len(payload)

    candidate_model = candidate_model.to(DEVICE).eval()
    clean_result = evaluate(candidate_model)

    expected_clean = float(
        c6.loc[c6["candidate_id"] == candidate_id, "accuracy_percent"].iloc[0]
    )

    if abs(clean_result["accuracy"] - expected_clean) > 1e-9:
        raise RuntimeError(
            f"{candidate_id} clean reproduction mismatch: "
            f"{clean_result['accuracy']} vs {expected_clean}"
        )

    return {
        "payloads": payloads,
        "total_bytes": total_bytes,
        "clean_accuracy": clean_result["accuracy"],
        "clean_predictions": clean_result["predictions"],
        "allocation_audit": records
    }

# ------------------------------------------------------------
# 7. Prepare resumable trial file
# ------------------------------------------------------------

trial_path = OUT / "C7_global_fault_trials.csv"
audit_path = OUT / "C7_candidate_payload_audit.csv"
summary_path = OUT / "C7_fault_summary.csv"
manifest_path = OUT / "C7_experiment_manifest.json"

if trial_path.exists():
    old = pd.read_csv(trial_path)
else:
    old = pd.DataFrame()

completed = set()
if len(old):
    completed = {
        (str(r.candidate_id), int(r.fault_bits), int(r.realization))
        for r in old.itertuples(index=False)
        if str(r.status) == "SUCCESS"
    }

all_audits = []
new_rows = []

# ------------------------------------------------------------
# 8. Run global serialized-payload bit flips
# ------------------------------------------------------------

for candidate_id in CANDIDATES:
    print("\n" + "=" * 75)
    print("Preparing:", candidate_id)

    built = build_candidate(candidate_id)
    payloads = built["payloads"]
    total_bytes = built["total_bytes"]
    total_bits = total_bytes * 8

    print("Payload bytes:", total_bytes)
    print("Clean accuracy:", built["clean_accuracy"])

    for rec in built["allocation_audit"]:
        rec["candidate_id"] = candidate_id
        all_audits.append(rec)

    for fault_bits in FAULT_COUNTS:
        for realization in range(1, TRIALS_PER_SETTING + 1):

            key = (candidate_id, fault_bits, realization)
            if key in completed:
                continue

            seed = BASE_SEED + (
                CANDIDATES.index(candidate_id) * 1_000_000
                + fault_bits * 100_000
                + realization
            )

            rng = np.random.default_rng(seed)
            selected = rng.choice(
                total_bits,
                size=fault_bits,
                replace=False
            ).astype(np.int64)

            # Copy payloads independently for this trial.
            corrupted = [
                bytearray(item["payload"])
                for item in payloads
            ]

            # Map each global bit position into its layer payload.
            affected = {}

            for global_bit in selected:
                global_byte = int(global_bit // 8)
                bit_in_byte = int(global_bit % 8)

                owner_index = None
                for i, item in enumerate(payloads):
                    if item["start_byte"] <= global_byte < item["end_byte"]:
                        owner_index = i
                        break

                if owner_index is None:
                    raise RuntimeError("Could not map global bit to layer.")

                item = payloads[owner_index]
                local_byte = global_byte - item["start_byte"]

                corrupted[owner_index][local_byte] ^= (1 << bit_in_byte)

                affected.setdefault(item["layer"], {
                    "format": item["format"],
                    "flips": 0
                })
                affected[item["layer"]]["flips"] += 1

            # Fresh FP32 model copy for every fault trial.
            fault_model = copy.deepcopy(base_model).cpu().eval()
            modules = dict(fault_model.named_modules())

            nonfinite = 0
            for i, item in enumerate(payloads):
                module = modules[item["layer"]]

                decoded = np.asarray(decode_weights(
        np.frombuffer(payload, dtype=np.uint8).copy(),
        metadata,
        fmt))

                nonfinite += int(np.size(decoded) - np.isfinite(decoded).sum())

                if not np.isfinite(decoded).all():
                    break

                module.weight.data.copy_(
                    torch.from_numpy(decoded).to(dtype=module.weight.dtype)
                )

            if nonfinite:
                row = {
                    "candidate_id": candidate_id,
                    "fault_bits": fault_bits,
                    "realization": realization,
                    "seed": seed,
                    "selected_bit_positions": json.dumps(selected.tolist()),
                    "clean_accuracy_percent": built["clean_accuracy"],
                    "fault_accuracy_percent": np.nan,
                    "accuracy_degradation_pp": np.nan,
                    "changed_predictions": np.nan,
                    "correct_to_wrong": np.nan,
                    "wrong_to_correct": np.nan,
                    "affected_layers": json.dumps(affected),
                    "nonfinite_weights": nonfinite,
                    "status": "NONFINITE",
                    "error_message": "Non-finite decoded weights"
                }
            else:
                fault_model = fault_model.to(DEVICE).eval()
                result = evaluate(fault_model)

                clean_pred = built["clean_predictions"]
                fault_pred = result["predictions"]

                changed = int(np.sum(clean_pred != fault_pred))
                clean_correct = clean_pred == ref_labels
                fault_correct = fault_pred == ref_labels

                c_to_w = int(np.sum(clean_correct & ~fault_correct))
                w_to_c = int(np.sum(~clean_correct & fault_correct))

                degradation = (
                    built["clean_accuracy"] - result["accuracy"]
                )

                row = {
                    "candidate_id": candidate_id,
                    "fault_bits": fault_bits,
                    "realization": realization,
                    "seed": seed,
                    "selected_bit_positions": json.dumps(selected.tolist()),
                    "clean_payload_sha256": hashlib.sha256(
                        b"".join(item["payload"] for item in payloads)
                    ).hexdigest(),
                    "clean_accuracy_percent": built["clean_accuracy"],
                    "fault_accuracy_percent": result["accuracy"],
                    "accuracy_degradation_pp": degradation,
                    "changed_predictions": changed,
                    "correct_to_wrong": c_to_w,
                    "wrong_to_correct": w_to_c,
                    "affected_layers": json.dumps(affected),
                    "nonfinite_weights": 0,
                    "status": "SUCCESS",
                    "error_message": ""
                }

                del fault_model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            new_rows.append(row)

            # Persist after every trial for safe resume.
            combined = pd.concat(
                [old, pd.DataFrame(new_rows)],
                ignore_index=True
            )
            combined.to_csv(trial_path, index=False)
            old = combined
            new_rows = []
            completed.add(key)

            if realization % 10 == 0:
                print(
                    f"{candidate_id} | {fault_bits}-bit | "
                    f"{realization}/{TRIALS_PER_SETTING} complete"
                )

    del built
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# 9. Summarize results
# ------------------------------------------------------------

trials = pd.read_csv(trial_path)
successful = trials[trials["status"] == "SUCCESS"].copy()

if len(successful) == 0:
    raise RuntimeError("No successful C7 fault trials.")

summaries = []

for (candidate_id, fault_bits), group in successful.groupby(
    ["candidate_id", "fault_bits"]
):
    x = group["accuracy_degradation_pp"].astype(float)

    summaries.append({
        "candidate_id": candidate_id,
        "fault_bits": int(fault_bits),
        "successful_trials": len(group),
        "mean_degradation_pp": x.mean(),
        "median_degradation_pp": x.median(),
        "p95_degradation_pp": x.quantile(0.95),
        "worst_observed_degradation_pp": x.max(),
        "best_observed_degradation_pp": x.min(),
        "catastrophic_trials_ge_10pp": int((x >= 10).sum()),
        "catastrophic_rate_percent": 100 * float((x >= 10).mean()),
        "mean_changed_predictions": group["changed_predictions"].mean(),
        "total_correct_to_wrong": int(group["correct_to_wrong"].sum()),
        "total_wrong_to_correct": int(group["wrong_to_correct"].sum()),
        "nonfinite_trials": int((group["nonfinite_weights"] > 0).sum())
    })

summary = pd.DataFrame(summaries)

summary.to_csv(summary_path, index=False)

if all_audits:
    pd.DataFrame(all_audits).drop_duplicates(
        ["candidate_id", "layer"]
    ).to_csv(audit_path, index=False)

manifest = {
    "stage": "C7",
    "purpose": "Extended full-model global serialized-payload fault validation",
    "candidates": CANDIDATES,
    "fault_counts": FAULT_COUNTS,
    "planned_trials_per_candidate_per_fault_count": TRIALS_PER_SETTING,
    "planned_new_trials": (
        len(CANDIDATES) * len(FAULT_COUNTS) * TRIALS_PER_SETTING
    ),
    "seed_base": BASE_SEED,
    "fault_sampling": (
        "Uniform random bit positions from concatenated serialized payload; "
        "without replacement within each trial."
    ),
    "comparison": (
        "Each faulted candidate is compared against its own clean candidate "
        "predictions and accuracy."
    ),
    "checkpoint_sha256": sha256_file(checkpoint_path),
    "reference_samples": 5000,
    "codec_functions_loaded": sorted(found),
    "successful_trials": int((trials["status"] == "SUCCESS").sum()),
    "nonfinite_trials": int((trials["status"] == "NONFINITE").sum()),
    "outputs": {
        "trials": str(trial_path),
        "summary": str(summary_path),
        "payload_audit": str(audit_path)
    },
    "runtime_seconds": round(time.time() - START, 2)
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# ------------------------------------------------------------
# 10. Final report
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("C7 FAULT VALIDATION SUMMARY")
print("=" * 90)
print(summary.to_string(index=False))

print("\nTrial counts:")
print(
    trials.groupby(["candidate_id", "fault_bits", "status"])
    .size().to_string()
)

print("\nArtifacts:", OUT)
print("Runtime:", round(time.time() - START, 2), "seconds")
print("C7 COMPLETE")

In [ ]:

from pathlib import Path
import os, json, ast, time, copy, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

# ============================================================
# JFAMP C7 — EXTENDED FULL-MODEL GLOBAL FAULT VALIDATION
# CORRECTED CODEC INPUT AND FAULT-DECODING LOGIC
# ============================================================

INPUT = Path("/kaggle/input")
ROOT = INPUT / "notebooks/crahulnaidu/mobile-net-100-5/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware"

C2_DIR = ROOT / "stage_C2_multiobjective_optimizer"

C6_DIR = Path(
    "/kaggle/working/JFAMP_MobileNetV2/experiments/"
    "revised_jfamp_fault_aware/stage_C6_corrected_clean_validation"
)
C6_RESULT = C6_DIR / "C6R_clean_candidate_results.csv"

OUT = Path(
    "/kaggle/working/JFAMP_MobileNetV2/experiments/"
    "revised_jfamp_fault_aware/stage_C7_extended_fault_validation"
)
OUT.mkdir(parents=True, exist_ok=True)

CANDIDATES = ["CAND_00853", "CAND_00361", "CAND_01218"]
FAULT_COUNTS = [1, 8]
TRIALS_PER_SETTING = 100
BASE_SEED = 2026111600

CHECKPOINT_NAME = "mobilenetv2_cifar100_best.pth"
REFERENCE_NAME = "fp32_reference_predictions.npz"
CHECKPOINT_SHA256 = "07c3c343451dcef593efc9172b9e07194b72f7c8c635b115f03104d7a8b2ff0a"

BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FP32_WEIGHT_BYTES = 9_271_040

START = time.time()

print("=" * 90)
print("JFAMP C7 — EXTENDED GLOBAL FAULT VALIDATION")
print("=" * 90)
print("Device:", DEVICE)


# ------------------------------------------------------------
# Utility functions
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def decode_serialized_payload(payload, metadata, fmt):
    """
    The validated decoder expects a writable NumPy uint8 array,
    not Python bytes.

    np.frombuffer preserves the exact serialized byte sequence.
    .copy() makes the array writable.
    """
    payload_array = np.frombuffer(
        bytes(payload),
        dtype=np.uint8
    ).copy()

    decoded = decode_weights(
        payload_array,
        metadata,
        fmt
    )

    return np.asarray(decoded)


# ------------------------------------------------------------
# 1. Verify C6R clean validation
# ------------------------------------------------------------

if not C6_RESULT.is_file():
    raise RuntimeError("C6R results missing. Complete corrected C6 first.")

c6 = pd.read_csv(C6_RESULT)

for candidate in CANDIDATES:
    row = c6[c6["candidate_id"] == candidate]

    if len(row) != 1:
        raise RuntimeError(f"Missing C6R result for {candidate}")

    if not bool(row.iloc[0]["overall_clean_feasible"]):
        raise RuntimeError(f"{candidate} is not clean-feasible.")

print("\nC6R candidate feasibility verified.")


# ------------------------------------------------------------
# 2. Locate canonical checkpoint and reference
# ------------------------------------------------------------

checkpoints = [
    p for p in INPUT.rglob(CHECKPOINT_NAME)
    if p.is_file() and sha256_file(p) == CHECKPOINT_SHA256
]

if not checkpoints:
    raise RuntimeError("Canonical checkpoint missing or SHA mismatch.")

preferred_checkpoints = [
    p for p in checkpoints
    if "mobile-net-100-3" in str(p)
]

checkpoint_path = (
    preferred_checkpoints[0]
    if preferred_checkpoints
    else checkpoints[0]
)

REQUIRED_REFERENCE_KEYS = {
    "logits": (5000, 100),
    "predictions": (5000,),
    "labels": (5000,),
    "correct": (5000,),
    "validation_indices": (5000,)
}

valid_references = []
rejected_references = []

for p in INPUT.rglob(REFERENCE_NAME):
    try:
        with np.load(p, allow_pickle=False) as z:
            available = set(z.files)

            missing = set(REQUIRED_REFERENCE_KEYS) - available

            if missing:
                rejected_references.append(
                    (str(p), f"Missing keys: {sorted(missing)}")
                )
                continue

            shape_errors = []

            for key, expected_shape in REQUIRED_REFERENCE_KEYS.items():
                if z[key].shape != expected_shape:
                    shape_errors.append(
                        f"{key}: {z[key].shape}, expected {expected_shape}"
                    )

            if shape_errors:
                rejected_references.append(
                    (str(p), "; ".join(shape_errors))
                )
                continue

            valid_references.append(p)

    except Exception as e:
        rejected_references.append((str(p), str(e)))

if not valid_references:
    print("Rejected reference files:")
    for path, reason in rejected_references:
        print(path, "->", reason)

    raise RuntimeError(
        "No canonical reference archive with all required arrays was found."
    )

preferred = [
    p for p in valid_references
    if "mobile-net-100-3" in str(p)
    and "stage_02_baseline" in str(p)
]

reference_path = (
    preferred[0] if preferred else valid_references[0]
)

with np.load(reference_path, allow_pickle=False) as ref:
    indices = ref["validation_indices"].astype(np.int64)
    ref_predictions = ref["predictions"].astype(np.int64)
    ref_labels = ref["labels"].astype(np.int64)
    ref_logits = ref["logits"].astype(np.float32)
    ref_correct = ref["correct"].astype(bool)

if len(np.unique(indices)) != 5000:
    raise RuntimeError("Canonical validation indices are not unique.")

if not np.array_equal(ref_correct, ref_predictions == ref_labels):
    raise RuntimeError("Reference correct mask is inconsistent.")

print("\nCanonical checkpoint:", checkpoint_path)
print("Checkpoint SHA256:", sha256_file(checkpoint_path))
print("Canonical reference:", reference_path)
print("Validation samples:", len(indices))
print("Unique validation indices:", len(np.unique(indices)))
print("Reference accuracy:", 100 * ref_correct.mean())


# ------------------------------------------------------------
# 3. Reconstruct canonical validation data
# ------------------------------------------------------------

train_files = [
    p for p in INPUT.rglob("cifar-100-python/train")
    if p.is_file()
]

if not train_files:
    raise RuntimeError("CIFAR-100 training file not found.")

data_root = train_files[0].parent.parent

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

full_dataset = datasets.CIFAR100(
    root=str(data_root),
    train=True,
    download=False,
    transform=eval_transform
)

val_loader = DataLoader(
    Subset(full_dataset, indices.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Validation loader reconstructed:", len(val_loader.dataset))


# ------------------------------------------------------------
# 4. Load exact validated codec functions
# ------------------------------------------------------------

required_functions = {
    "encode_weights",
    "decode_weights",
    "encode_e4m11",
    "decode_e4m11",
    "pack_int4",
    "unpack_int4"
}

namespace = {
    "np": np,
    "torch": torch,
    "nn": nn
}

found = set()

for path in list(INPUT.rglob("*.py")) + list(INPUT.rglob("*.ipynb")):

    try:
        if path.suffix == ".py":
            sources = [path.read_text(errors="ignore")]

        else:
            nb = json.loads(path.read_text(errors="ignore"))
            sources = [
                "".join(c.get("source", []))
                for c in nb.get("cells", [])
                if c.get("cell_type") == "code"
            ]

        for source in sources:

            try:
                tree = ast.parse(source)
            except Exception:
                continue

            defs = {
                n.name
                for n in tree.body
                if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
            }

            matched = defs & required_functions

            if matched:
                exec(
                    compile(source, str(path), "exec"),
                    namespace
                )
                found.update(matched)

    except Exception:
        continue

missing = required_functions - found

if missing:
    raise RuntimeError(
        "Missing exact codec functions: " + str(sorted(missing))
    )

encode_weights = namespace["encode_weights"]
decode_weights = namespace["decode_weights"]

print("\nCodec functions loaded:", sorted(found))


# ------------------------------------------------------------
# 5. Load immutable FP32 model and verify reference
# ------------------------------------------------------------

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

base_model = models.mobilenet_v2(
    weights=None,
    num_classes=100
)

base_model.features[0][0].stride = (1, 1)

base_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

base_model = base_model.to(DEVICE).eval()


def evaluate(model):
    model.eval()

    predictions = []
    labels = []

    with torch.inference_mode():
        for images, target in val_loader:

            logits = model(
                images.to(DEVICE, non_blocking=True)
            )

            predictions.append(
                logits.argmax(1).cpu().numpy()
            )

            labels.append(target.numpy())

    predictions = np.concatenate(predictions)
    labels = np.concatenate(labels)

    return {
        "predictions": predictions,
        "labels": labels,
        "accuracy": float(
            100 * np.mean(predictions == labels)
        ),
        "correct_count": int(
            np.sum(predictions == labels)
        )
    }


fp32 = evaluate(base_model)

if not np.array_equal(fp32["predictions"], ref_predictions):
    raise RuntimeError("FP32 reference predictions do not match.")

if not np.array_equal(fp32["labels"], ref_labels):
    raise RuntimeError("FP32 labels do not match.")

print(
    f"\nFP32 baseline reproduced: "
    f"{fp32['accuracy']:.4f}%"
)


# ------------------------------------------------------------
# 6. Load candidate allocations
# ------------------------------------------------------------

allocation_file = C2_DIR / "C2_candidate_layer_allocations.csv"

if not allocation_file.is_file():
    raise RuntimeError(f"Missing allocation file: {allocation_file}")

allocations = pd.read_csv(allocation_file)


# ------------------------------------------------------------
# 7. Build each candidate's exact serialized payload
# ------------------------------------------------------------

def build_candidate(candidate_id):

    group = allocations[
        allocations["candidate_id"] == candidate_id
    ].copy()

    if len(group) != 53 or group["layer"].nunique() != 53:
        raise RuntimeError(
            f"Invalid allocation for {candidate_id}"
        )

    # Create a fresh candidate model before looking up its modules.
    candidate_model = copy.deepcopy(base_model).cpu().eval()
    modules = dict(candidate_model.named_modules())

    payloads = []
    records = []

    offset = 0
    total_bytes = 0

    for row in group.itertuples(index=False):

        layer = str(row.layer)
        fmt = str(row.format)

        if layer not in modules:
            raise RuntimeError(f"Missing module {layer}")

        module = modules[layer]

        original = (
            module.weight.detach()
            .cpu()
            .numpy()
            .copy()
        )

        payload, metadata = encode_weights(original, fmt)
        payload = bytes(payload)

        if len(payload) != int(row.payload_bytes):
            raise RuntimeError(
                f"Payload size mismatch: {layer}/{fmt}"
            )

        # FIX 1: decode accepts writable uint8 NumPy arrays.
        decoded = decode_serialized_payload(
            payload,
            metadata,
            fmt
        )

        if decoded.shape != original.shape:
            raise RuntimeError(
                f"Decoded shape mismatch: {layer}/{fmt}"
            )

        if not np.isfinite(decoded).all():
            raise RuntimeError(
                f"Non-finite clean decode: {layer}/{fmt}"
            )

        module.weight.data.copy_(
            torch.from_numpy(
                np.array(decoded, copy=True)
            ).to(dtype=module.weight.dtype)
        )

        payloads.append({
            "layer": layer,
            "format": fmt,
            "payload": payload,
            "metadata": metadata,
            "start_byte": offset,
            "end_byte": offset + len(payload)
        })

        records.append({
            "layer": layer,
            "format": fmt,
            "payload_bytes": len(payload),
            "payload_sha256": hashlib.sha256(
                payload
            ).hexdigest()
        })

        offset += len(payload)
        total_bytes += len(payload)

    candidate_model = candidate_model.to(DEVICE).eval()

    clean_result = evaluate(candidate_model)

    expected_clean = float(
        c6.loc[
            c6["candidate_id"] == candidate_id,
            "accuracy_percent"
        ].iloc[0]
    )

    if abs(clean_result["accuracy"] - expected_clean) > 1e-9:
        raise RuntimeError(
            f"{candidate_id} clean reproduction mismatch: "
            f"{clean_result['accuracy']} vs {expected_clean}"
        )

    payload_hash = hashlib.sha256(
        b"".join(item["payload"] for item in payloads)
    ).hexdigest()

    print(
        f"{candidate_id}: clean={clean_result['accuracy']:.4f}%, "
        f"payload={total_bytes:,} bytes"
    )

    del candidate_model

    return {
        "payloads": payloads,
        "total_bytes": total_bytes,
        "clean_accuracy": clean_result["accuracy"],
        "clean_predictions": clean_result["predictions"],
        "allocation_audit": records,
        "payload_sha256": payload_hash
    }


# ------------------------------------------------------------
# 8. Prepare resumable trial files
# ------------------------------------------------------------

trial_path = OUT / "C7_global_fault_trials.csv"
audit_path = OUT / "C7_candidate_payload_audit.csv"
summary_path = OUT / "C7_fault_summary.csv"
manifest_path = OUT / "C7_experiment_manifest.json"

if trial_path.exists():
    old = pd.read_csv(trial_path)
else:
    old = pd.DataFrame()

completed = set()

if len(old):
    completed = {
        (
            str(r.candidate_id),
            int(r.fault_bits),
            int(r.realization)
        )
        for r in old.itertuples(index=False)
        if str(r.status) == "SUCCESS"
    }

print("\nPreviously completed successful trials:", len(completed))

all_audits = []
new_rows = []


# ------------------------------------------------------------
# 9. Run global serialized-payload bit flips
# ------------------------------------------------------------

for candidate_id in CANDIDATES:

    print("\n" + "=" * 75)
    print("Preparing:", candidate_id)

    built = build_candidate(candidate_id)

    payloads = built["payloads"]
    total_bytes = built["total_bytes"]
    total_bits = total_bytes * 8

    print("Payload bytes:", total_bytes)
    print("Payload bits:", total_bits)
    print("Clean accuracy:", built["clean_accuracy"])
    print("Clean payload SHA256:", built["payload_sha256"])

    for rec in built["allocation_audit"]:
        rec["candidate_id"] = candidate_id
        all_audits.append(rec)

    for fault_bits in FAULT_COUNTS:

        for realization in range(1, TRIALS_PER_SETTING + 1):

            key = (
                candidate_id,
                fault_bits,
                realization
            )

            if key in completed:
                continue

            seed = BASE_SEED + (
                CANDIDATES.index(candidate_id) * 1_000_000
                + fault_bits * 100_000
                + realization
            )

            rng = np.random.default_rng(seed)

            selected = rng.choice(
                total_bits,
                size=fault_bits,
                replace=False
            ).astype(np.int64)

            # Make independent mutable copies of each payload.
            corrupted = [
                bytearray(item["payload"])
                for item in payloads
            ]

            # Map global bit positions to individual layers.
            affected = {}

            for global_bit in selected:

                global_byte = int(global_bit // 8)
                bit_in_byte = int(global_bit % 8)

                owner_index = None

                for i, item in enumerate(payloads):

                    if (
                        item["start_byte"]
                        <= global_byte
                        < item["end_byte"]
                    ):
                        owner_index = i
                        break

                if owner_index is None:
                    raise RuntimeError(
                        "Could not map global bit to layer."
                    )

                item = payloads[owner_index]

                local_byte = global_byte - item["start_byte"]

                corrupted[owner_index][local_byte] ^= (
                    1 << bit_in_byte
                )

                affected.setdefault(
                    item["layer"],
                    {
                        "format": item["format"],
                        "flips": 0
                    }
                )

                affected[item["layer"]]["flips"] += 1

            # Fresh FP32 model copy for every fault trial.
            fault_model = copy.deepcopy(
                base_model
            ).cpu().eval()

            modules = dict(fault_model.named_modules())

            nonfinite = 0

            # Decode every layer using its own corrupted payload.
            for i, item in enumerate(payloads):

                module = modules[item["layer"]]

                # FIX 2:
                # Use corrupted[i], item["metadata"], item["format"].
                # The previous code incorrectly referenced stale or
                # undefined payload / metadata / fmt variables.
                decoded = decode_serialized_payload(
                    corrupted[i],
                    item["metadata"],
                    item["format"]
                )

                nonfinite += int(
                    decoded.size - np.isfinite(decoded).sum()
                )

                if not np.isfinite(decoded).all():
                    break

                if decoded.shape != tuple(module.weight.shape):
                    raise RuntimeError(
                        f"Fault decode shape mismatch: {item['layer']}"
                    )

                module.weight.data.copy_(
                    torch.from_numpy(
                        np.array(decoded, copy=True)
                    ).to(dtype=module.weight.dtype)
                )

            if nonfinite:

                row = {
                    "candidate_id": candidate_id,
                    "fault_bits": fault_bits,
                    "realization": realization,
                    "seed": seed,
                    "selected_bit_positions": json.dumps(
                        selected.tolist()
                    ),
                    "clean_payload_sha256": built["payload_sha256"],
                    "clean_accuracy_percent": built["clean_accuracy"],
                    "fault_accuracy_percent": np.nan,
                    "accuracy_degradation_pp": np.nan,
                    "changed_predictions": np.nan,
                    "correct_to_wrong": np.nan,
                    "wrong_to_correct": np.nan,
                    "affected_layers": json.dumps(affected),
                    "nonfinite_weights": nonfinite,
                    "status": "NONFINITE",
                    "error_message": "Non-finite decoded weights"
                }

            else:

                fault_model = fault_model.to(DEVICE).eval()

                result = evaluate(fault_model)

                clean_pred = built["clean_predictions"]
                fault_pred = result["predictions"]

                changed = int(
                    np.sum(clean_pred != fault_pred)
                )

                clean_correct = clean_pred == ref_labels
                fault_correct = fault_pred == ref_labels

                c_to_w = int(
                    np.sum(clean_correct & ~fault_correct)
                )

                w_to_c = int(
                    np.sum(~clean_correct & fault_correct)
                )

                degradation = (
                    built["clean_accuracy"] - result["accuracy"]
                )

                row = {
                    "candidate_id": candidate_id,
                    "fault_bits": fault_bits,
                    "realization": realization,
                    "seed": seed,
                    "selected_bit_positions": json.dumps(
                        selected.tolist()
                    ),
                    "clean_payload_sha256": built["payload_sha256"],
                    "clean_accuracy_percent": built["clean_accuracy"],
                    "fault_accuracy_percent": result["accuracy"],
                    "accuracy_degradation_pp": degradation,
                    "changed_predictions": changed,
                    "correct_to_wrong": c_to_w,
                    "wrong_to_correct": w_to_c,
                    "affected_layers": json.dumps(affected),
                    "nonfinite_weights": 0,
                    "status": "SUCCESS",
                    "error_message": ""
                }

            # Release model resources before continuing.
            del fault_model

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            new_rows.append(row)

            # Persist after every trial for safe resume.
            combined = pd.concat(
                [old, pd.DataFrame(new_rows)],
                ignore_index=True
            )

            combined.to_csv(
                trial_path,
                index=False
            )

            old = combined
            new_rows = []

            completed.add(key)

            if realization % 10 == 0:
                print(
                    f"{candidate_id} | "
                    f"{fault_bits}-bit | "
                    f"{realization}/{TRIALS_PER_SETTING} complete"
                )

    del built

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# 10. Summarize results
# ------------------------------------------------------------

if not trial_path.exists():
    raise RuntimeError("No trial file was generated.")

trials = pd.read_csv(trial_path)

successful = trials[
    trials["status"] == "SUCCESS"
].copy()

if len(successful) == 0:
    raise RuntimeError("No successful C7 fault trials.")

summaries = []

for (candidate_id, fault_bits), group in successful.groupby(
    ["candidate_id", "fault_bits"]
):

    x = group["accuracy_degradation_pp"].astype(float)

    summaries.append({
        "candidate_id": candidate_id,
        "fault_bits": int(fault_bits),
        "successful_trials": len(group),
        "mean_degradation_pp": x.mean(),
        "median_degradation_pp": x.median(),
        "p95_degradation_pp": x.quantile(0.95),
        "worst_observed_degradation_pp": x.max(),
        "best_observed_degradation_pp": x.min(),
        "catastrophic_trials_ge_10pp": int(
            (x >= 10).sum()
        ),
        "catastrophic_rate_percent": 100 * float(
            (x >= 10).mean()
        ),
        "mean_changed_predictions": group[
            "changed_predictions"
        ].mean(),
        "total_correct_to_wrong": int(
            group["correct_to_wrong"].sum()
        ),
        "total_wrong_to_correct": int(
            group["wrong_to_correct"].sum()
        ),
        "nonfinite_trials": int(
            (group["nonfinite_weights"] > 0).sum()
        )
    })

summary = pd.DataFrame(summaries)

summary.to_csv(
    summary_path,
    index=False
)

if all_audits:

    pd.DataFrame(all_audits).drop_duplicates(
        ["candidate_id", "layer"]
    ).to_csv(
        audit_path,
        index=False
    )


# ------------------------------------------------------------
# 11. Experiment manifest
# ------------------------------------------------------------

manifest = {
    "stage": "C7",
    "purpose": (
        "Extended full-model global serialized-payload "
        "fault validation"
    ),
    "candidates": CANDIDATES,
    "fault_counts": FAULT_COUNTS,
    "planned_trials_per_candidate_per_fault_count": (
        TRIALS_PER_SETTING
    ),
    "planned_new_trials": (
        len(CANDIDATES)
        * len(FAULT_COUNTS)
        * TRIALS_PER_SETTING
    ),
    "seed_base": BASE_SEED,
    "fault_sampling": (
        "Uniform random bit positions from concatenated "
        "serialized payload; without replacement within each trial."
    ),
    "comparison": (
        "Each faulted candidate is compared against its own "
        "clean candidate predictions and accuracy."
    ),
    "checkpoint_sha256": sha256_file(checkpoint_path),
    "reference_path": str(reference_path),
    "reference_samples": 5000,
    "reference_accuracy_percent": float(
        100 * ref_correct.mean()
    ),
    "codec_functions_loaded": sorted(found),
    "successful_trials": int(
        (trials["status"] == "SUCCESS").sum()
    ),
    "nonfinite_trials": int(
        (trials["status"] == "NONFINITE").sum()
    ),
    "trial_status_counts": {
        str(k): int(v)
        for k, v in trials["status"].value_counts().items()
    },
    "outputs": {
        "trials": str(trial_path),
        "summary": str(summary_path),
        "payload_audit": str(audit_path)
    },
    "runtime_seconds": round(
        time.time() - START,
        2
    )
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)


# ------------------------------------------------------------
# 12. Final report
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("C7 FAULT VALIDATION SUMMARY")
print("=" * 90)

print(summary.to_string(index=False))

print("\nTrial counts:")

print(
    trials.groupby(
        ["candidate_id", "fault_bits", "status"]
    ).size().to_string()
)

print("\nExpected new successful trials: 600")
print(
    "Actual new successful trials:",
    int((trials["status"] == "SUCCESS").sum())
)

print("\nArtifacts:", OUT)
print("Runtime:", round(time.time() - START, 2), "seconds")
print("C7 COMPLETE")


In [1]:
import os
import pandas as pd

INPUT_DIR = "/kaggle/input"

keywords = [
    "C5_selected_candidate_layer_allocations.csv",
    "C6R_clean_candidate_results.csv",
    "C7_fault_summary.csv",
    "C7_candidate_payload_audit.csv",
    "C7_experiment_manifest.json"
]

found = {}

for root, dirs, files in os.walk(INPUT_DIR):
    for filename in files:
        if filename in keywords:
            found[filename] = os.path.join(root, filename)

print("=" * 80)
print("DISCOVERED EXPERIMENT FILES")
print("=" * 80)

for name, path in found.items():
    print(f"\n{name}")
    print(path)

    if name.endswith(".csv"):
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(3).to_string(index=False))

DISCOVERED EXPERIMENT FILES


In [2]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

# ---------------------------------------------------------
# 1. Locate experiment files
# ---------------------------------------------------------

INPUT_DIR = "/kaggle/input"

def find_file(filename):
    matches = []
    for root, dirs, files in os.walk(INPUT_DIR):
        if filename in files:
            matches.append(os.path.join(root, filename))

    if not matches:
        raise FileNotFoundError(f"Could not find {filename}")

    if len(matches) > 1:
        print(f"Multiple copies of {filename} found:")
        for p in matches:
            print(" ", p)
        print("Using the first match.")

    return matches[0]


fault_path = find_file("C7_fault_summary.csv")
allocation_path = find_file("C5_selected_candidate_layer_allocations.csv")

fault = pd.read_csv(fault_path)
alloc = pd.read_csv(allocation_path)

print("Fault summary:", fault_path)
print("Allocation file:", allocation_path)

# ---------------------------------------------------------
# 2. Identify required columns
# ---------------------------------------------------------

def get_column(df, possible_names):
    lookup = {c.lower().strip(): c for c in df.columns}
    for name in possible_names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None


candidate_col = get_column(fault, ["candidate_id", "candidate"])
bits_col = get_column(fault, ["fault_bits", "num_fault_bits", "bits"])
degradation_col = get_column(
    fault,
    ["mean_degradation_pp", "mean_accuracy_degradation_pp",
     "mean_accuracy_drop_pp"]
)

if not all([candidate_col, bits_col, degradation_col]):
    raise ValueError(
        "Could not identify required columns in C7 summary.\n"
        f"Available columns: {fault.columns.tolist()}"
    )

# ---------------------------------------------------------
# 3. Calculate equal-weighted fault score
# ---------------------------------------------------------

fault[bits_col] = pd.to_numeric(fault[bits_col], errors="coerce")
fault[degradation_col] = pd.to_numeric(
    fault[degradation_col], errors="coerce"
)

pivot = fault.pivot_table(
    index=candidate_col,
    columns=bits_col,
    values=degradation_col,
    aggfunc="mean"
)

if 1 not in pivot.columns or 8 not in pivot.columns:
    raise ValueError("Both 1-bit and 8-bit results are required.")

ranking = pd.DataFrame(index=pivot.index)
ranking["mean_1bit_degradation_pp"] = pivot[1]
ranking["mean_8bit_degradation_pp"] = pivot[8]

ranking["combined_fault_score_pp"] = (
    ranking["mean_1bit_degradation_pp"]
    + ranking["mean_8bit_degradation_pp"]
) / 2

# Include catastrophic trial rates if available
cat_col = get_column(
    fault,
    ["catastrophic_rate_percent", "catastrophic_trials_ge_10pp"]
)

if cat_col:
    cat = fault.pivot_table(
        index=candidate_col,
        columns=bits_col,
        values=cat_col,
        aggfunc="mean"
    )
    if 1 in cat.columns:
        ranking["catastrophic_1bit"] = cat[1]
    if 8 in cat.columns:
        ranking["catastrophic_8bit"] = cat[8]

ranking = ranking.sort_values("combined_fault_score_pp")

print("\n" + "=" * 80)
print("CANDIDATE RANKING — LOWER FAULT SCORE IS BETTER")
print("=" * 80)

display(ranking)

best_candidate = ranking.index[0]

print("\nBest candidate by equally weighted mean fault degradation:")
print(best_candidate)

# ---------------------------------------------------------
# 4. Extract the exact layer-wise assignment
# ---------------------------------------------------------

allocation_candidate_col = get_column(
    alloc, ["candidate_id", "candidate"]
)

layer_col = get_column(
    alloc, ["layer", "layer_name", "layer_id", "module"]
)

datatype_col = get_column(
    alloc, ["datatype", "data_type", "format", "selected_format"]
)

if not all([allocation_candidate_col, layer_col, datatype_col]):
    raise ValueError(
        "Could not identify assignment columns.\n"
        f"Allocation columns: {alloc.columns.tolist()}"
    )

best_assignment = alloc[
    alloc[allocation_candidate_col].astype(str) == str(best_candidate)
].copy()

if best_assignment.empty:
    raise ValueError(
        f"{best_candidate} was not found in the allocation file. "
        "Check whether the allocation CSV contains all candidate assignments."
    )

best_assignment = best_assignment.sort_values(layer_col)

print("\n" + "=" * 80)
print(f"FINAL ASSIGNMENT FOR {best_candidate}")
print("=" * 80)

display(best_assignment[[layer_col, datatype_col]])

print("\nNumber of assigned layers:", len(best_assignment))
print("\nDatatype distribution:")
display(best_assignment[datatype_col].value_counts().to_frame("layer_count"))

# ---------------------------------------------------------
# 5. Save winning assignment
# ---------------------------------------------------------

output_path = "/kaggle/working/best_candidate_layerwise_allocation.csv"

best_assignment.to_csv(output_path, index=False)

print("\nSaved assignment to:", output_path)

# Download link
from IPython.display import FileLink
display(FileLink(output_path))

FileNotFoundError: Could not find C7_fault_summary.csv

In [1]:

# ============================================================
# JFAMP MobileNetV2 — DOWNLOAD ALL EXPERIMENT RESULTS
# Creates one ZIP containing the complete project workspace.
# Excludes only the downloaded CIFAR-100 image archive.
# ============================================================

from pathlib import Path
import zipfile
import os
from IPython.display import display, FileLink

# ------------------------------------------------------------
# 1. Project directories
# ------------------------------------------------------------

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")

ZIP_PATH = Path(
    "/kaggle/working/JFAMP_MobileNetV2_ALL_RESULTS.zip"
)

assert ROOT.exists(), f"Project directory not found: {ROOT}"

# ------------------------------------------------------------
# 2. Exclusion rules
# ------------------------------------------------------------

EXCLUDE_DIRS = {
    "torchvision",       # CIFAR-100 downloaded image archive
    "__pycache__",
    ".ipynb_checkpoints"
}

# ------------------------------------------------------------
# 3. Collect all project files
# ------------------------------------------------------------

all_files = []

for path in ROOT.rglob("*"):

    if not path.is_file():
        continue

    relative = path.relative_to(ROOT)

    # Exclude dataset cache and temporary directories.
    if any(part in EXCLUDE_DIRS for part in relative.parts):
        continue

    all_files.append(path)

all_files.sort()

print("=" * 75)
print("JFAMP MOBILENETV2 — RESULTS EXPORT")
print("=" * 75)

print("Project root:", ROOT)
print("Files discovered:", len(all_files))

# ------------------------------------------------------------
# 4. Create ZIP archive
# ------------------------------------------------------------

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as zipf:

    for path in all_files:

        relative = path.relative_to(ROOT)

        zipf.write(
            path,
            arcname=str(Path("JFAMP_MobileNetV2") / relative)
        )

# ------------------------------------------------------------
# 5. Verify archive
# ------------------------------------------------------------

with zipfile.ZipFile(ZIP_PATH, "r") as zipf:

    archived_files = zipf.namelist()

    assert len(archived_files) == len(all_files)

    bad_file = zipf.testzip()

    assert bad_file is None, f"Corrupt archive member: {bad_file}"

print("\nARCHIVE VERIFICATION")
print("Files archived:", len(archived_files))
print("ZIP integrity: PASSED")
print(
    "ZIP size:",
    round(ZIP_PATH.stat().st_size / (1024**2), 2),
    "MiB"
)

print("\nDOWNLOAD FILE:")
print(ZIP_PATH)

print("=" * 75)

# ------------------------------------------------------------
# 6. Display download link
# ------------------------------------------------------------

display(FileLink(str(ZIP_PATH)))


AssertionError: Project directory not found: /kaggle/working/JFAMP_MobileNetV2